In [ ]:
import os
import ast
import json
import math
import random
import warnings
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional, Tuple, Iterable
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, mannwhitneyu
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
try:
    from sklearn.linear_model import Ridge
except Exception:
    Ridge = None
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GINEConv, global_add_pool
try:
    from rdkit import Chem
    from rdkit.Chem import rdchem
except Exception as e:
    raise ImportError('RDKit is required. Install rdkit-pypi or conda rdkit.') from e
warnings.filterwarnings('ignore')

@dataclass
class ConfigDruXRV5:
    seed: int = 1
    seeds: Tuple[int, ...] = (1,)
    split_seeds: Tuple[int, ...] = (1, 7, 21, 42, 123, 256)
    split_seed: int = 1
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    output_dir: str = 'druxr_outputs_v5_r4'
    run_name: str = 'druxr_v5_clean_multispecialist'
    batch_size: int = 128
    epochs: int = 30
    lr: float = 0.0005
    weight_decay: float = 5e-05
    grad_clip_norm: float = 1.0
    expr_pca_dim: int = 256
    graph_hidden_dim: int = 64
    tab_hidden_dim: int = 128
    cell_hidden_dim: int = 128
    tic_hidden_dim: int = 64
    fusion_hidden_dim: int = 128
    dropout: float = 0.35
    min_moa_count: int = 3
    min_pathway_count: int = 3
    min_subtype_size: int = 10
    max_selective_subtypes_per_drug: int = 3
    selectivity_effect_threshold: float = 0.15
    selectivity_min_positive_drugs: int = 3
    rank_within_subtype: bool = True
    subtype_proxy_clip: float = 1.75
    rank_global_mix: float = 0.75
    train_frac: float = 0.8
    val_frac: float = 0.1
    test_frac: float = 0.1
    num_workers: int = 0
    use_amp: bool = True
    lambda_response: float = 2.0
    lambda_ln_ic50: float = 1.0
    lambda_tic: float = 0.5
    lambda_moa: float = 0.4
    lambda_selectivity: float = 0.2
    lambda_rank: float = 0.1
    lambda_response_pair_rank: float = 0.35
    lambda_response_global_rank: float = 0.1
    lambda_ln_ic50_pair_rank: float = 0.2
    lambda_ln_ic50_global_rank: float = 0.05
    lambda_pathway: float = 0.2
    lambda_domain_adv: float = 0.2
    tic_feature_dim: int = 24
    rank_min_delta_auc: float = 0.05
    rank_min_delta_ln_ic50: float = 0.25
    rank_max_pairs_per_group: int = 4096
    listwise_temperature: float = 0.15
    lambda_response_subtype_listwise: float = 0.1
    lambda_response_drug_listwise: float = 0.1
    lambda_ln_ic50_subtype_listwise: float = 0.05
    lambda_ln_ic50_drug_listwise: float = 0.05
    use_specialist_fusion: bool = True
    fusion_alpha: float = 1.0
    fusion_min_rows: int = 40
    fusion_min_improvement: float = -0.005
    posthoc_selectivity_delta_threshold: float = 0.05
    posthoc_selectivity_min_cells: int = 2
    specialist_name: str = 'combined'
    scheduler_factor: float = 0.5
    scheduler_patience: int = 2
    early_stop_patience: int = 6
    monitor_metric: str = 'response_discovery_composite'
    monitor_mode: str = 'max'
    aux_warmup_epochs: int = 5
    response_huber_beta: float = 0.5
    ln_ic50_huber_beta: float = 0.5
    tic_huber_beta: float = 0.5
    scale_drug_features: bool = True
    tic_min_target_genes: int = 2
    tic_min_neighbor_genes: int = 3
    allow_medium_only_domain_adv: bool = False
    remove_pathway_feature_leakage: bool = True
    valid_smiles_only: bool = True
    save_predictions: bool = True
    save_checkpoints: bool = True
    save_history: bool = True
    use_validation_calibration: bool = True
    clip_auc_predictions: bool = True
    topk_values: Tuple[int, ...] = (10, 25, 50, 100)
    discovery_top_n: int = 10
CFG_DRUXR_V5 = ConfigDruXRV5()

def _ensure_v5_config_defaults(cfg: ConfigDruXRV5) -> ConfigDruXRV5:
    return cfg
HYBRIDIZATION_MAP = {rdchem.HybridizationType.SP: 0, rdchem.HybridizationType.SP2: 1, rdchem.HybridizationType.SP3: 2, rdchem.HybridizationType.SP3D: 3, rdchem.HybridizationType.SP3D2: 4}
BOND_TYPE_MAP = {rdchem.BondType.SINGLE: 0, rdchem.BondType.DOUBLE: 1, rdchem.BondType.TRIPLE: 2, rdchem.BondType.AROMATIC: 3}

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def read_any_table(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    lower = path.lower()
    if lower.endswith('.parquet'):
        return pd.read_parquet(path)
    if lower.endswith('.pkl') or lower.endswith('.pickle'):
        return pd.read_pickle(path)
    if lower.endswith('.csv'):
        return pd.read_csv(path)
    raise ValueError(f'Unsupported input type: {path}')

def safe_split_pipe(x: object) -> List[str]:
    if pd.isna(x):
        return []
    if not isinstance(x, str):
        x = str(x)
    return [p.strip() for p in x.split('|') if p.strip()]

def parse_gene_list(x: object) -> List[str]:
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    s = str(x).strip()
    if not s or s.lower() in {'nan', 'none', '<na>'}:
        return []
    if s.startswith('[') and s.endswith(']'):
        try:
            out = ast.literal_eval(s)
            if isinstance(out, list):
                return [str(v).strip() for v in out if str(v).strip()]
        except Exception:
            pass
    for sep in ['|', ',', ';']:
        if sep in s:
            return [p.strip().strip("'").strip('"') for p in s.split(sep) if p.strip()]
    return [s]

def zscore_series(x: pd.Series) -> pd.Series:
    std = x.std(ddof=0)
    if pd.isna(std) or std == 0:
        return pd.Series(np.zeros(len(x)), index=x.index)
    return (x - x.mean()) / std

def minmax01(x: pd.Series) -> pd.Series:
    x_min, x_max = (x.min(), x.max())
    if pd.isna(x_min) or pd.isna(x_max) or x_max == x_min:
        return pd.Series(np.zeros(len(x)), index=x.index)
    return (x - x_min) / (x_max - x_min)

def first_nonempty_value(series: pd.Series):
    for v in series:
        if pd.notna(v) and str(v).strip() not in {'', 'nan', '<NA>', 'None'}:
            return v
    return pd.NA

def first_nonnull_value(series: pd.Series):
    for v in series:
        if pd.notna(v):
            return v
    return np.nan

def pick_first_available(df: pd.DataFrame, candidates: List[str], default=None):
    for c in candidates:
        if c in df.columns:
            return c
    return default

def choose_smiles_column(df: pd.DataFrame) -> Optional[str]:
    for c in ['smiles_best_v2', 'smiles_best', 'chembl_smiles', 'pubchem_isomeric_smiles', 'pubchem_canonical_smiles']:
        if c in df.columns:
            return c
    return None

def is_valid_smiles_value(smiles: object) -> bool:
    if pd.isna(smiles) or not str(smiles).strip():
        return False
    s = str(smiles).strip()
    if s.lower() in {'nan', 'none', '<na>'}:
        return False
    return Chem.MolFromSmiles(s) is not None

def build_domain_column(df: pd.DataFrame) -> pd.Series:
    if 'DOMAIN_COMBINED' in df.columns:
        return df['DOMAIN_COMBINED'].astype(str)
    bits = []
    for c in ['DATASET', 'SCREENING_SITE', 'SCREENING_MEDIUM', 'Screen Medium']:
        if c in df.columns:
            bits.append(df[c].fillna('UNKNOWN').astype(str))
    if not bits:
        return pd.Series(['UNKNOWN'] * len(df), index=df.index)
    out = bits[0]
    for b in bits[1:]:
        out = out + ' | ' + b
    return out

def detect_domain_mode(df: pd.DataFrame, allow_medium_only: bool=False) -> Dict[str, Any]:
    dataset_n = df['DATASET'].nunique(dropna=True) if 'DATASET' in df.columns else 0
    site_n = df['SCREENING_SITE'].nunique(dropna=True) if 'SCREENING_SITE' in df.columns else 0
    medium_col = 'SCREENING_MEDIUM' if 'SCREENING_MEDIUM' in df.columns else 'Screen Medium' if 'Screen Medium' in df.columns else None
    medium_n = df[medium_col].nunique(dropna=True) if medium_col is not None else 0
    true_multi = dataset_n > 1 or site_n > 1 or (allow_medium_only and medium_n > 1)
    medium_only = dataset_n <= 1 and site_n <= 1 and (medium_n > 1)
    return {'dataset_unique': int(dataset_n), 'site_unique': int(site_n), 'medium_unique': int(medium_n), 'medium_column': medium_col, 'true_multi_domain': bool(true_multi), 'medium_only_variation': bool(medium_only)}

def atom_features(atom: rdchem.Atom) -> List[float]:
    return [float(atom.GetAtomicNum()), float(atom.GetDegree()), float(atom.GetFormalCharge()), float(atom.GetTotalNumHs()), float(atom.GetIsAromatic()), float(HYBRIDIZATION_MAP.get(atom.GetHybridization(), 5)), float(atom.GetMass() * 0.01)]

def bond_features(bond: rdchem.Bond) -> List[float]:
    return [float(BOND_TYPE_MAP.get(bond.GetBondType(), 4)), float(bond.GetIsConjugated()), float(bond.IsInRing()), float(int(bond.GetStereo()))]

def smiles_to_pyg(smiles: str, drug_id: int) -> Optional[Data]:
    if pd.isna(smiles) or not str(smiles).strip():
        return None
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    edge_index, edge_attr = ([], [])
    for bond in mol.GetBonds():
        i, j = (bond.GetBeginAtomIdx(), bond.GetEndAtomIdx())
        bf = bond_features(bond)
        edge_index += [[i, j], [j, i]]
        edge_attr += [bf, bf]
    if edge_index:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, 4), dtype=torch.float)
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    data.drug_id = int(drug_id)
    return data

def build_graph_drug_dict(drug_df: pd.DataFrame) -> Tuple[Dict[int, Data], Optional[str]]:
    smiles_col = choose_smiles_column(drug_df)
    if smiles_col is None:
        return ({}, None)
    graph_dict, bad = ({}, [])
    for _, row in drug_df[['DRUG_ID', smiles_col]].iterrows():
        g = smiles_to_pyg(row[smiles_col], int(row['DRUG_ID']))
        if g is None:
            bad.append(int(row['DRUG_ID']))
        else:
            graph_dict[int(row['DRUG_ID'])] = g
    print(f'Molecular graphs built from {smiles_col}: {len(graph_dict)}')
    if bad:
        print(f'[SMILES] Failed graph parse count: {len(bad)}')
    return (graph_dict, smiles_col)

def infer_expression_columns(df: pd.DataFrame) -> List[str]:
    non_expr_known = {'COSMIC_ID', 'DRUG_ID', 'DRUG_NAME', 'COMPOUND_NAME', 'SYNONYMS', 'TARGET', 'PATHWAY_NAME', 'PUBCHEM', 'CELL_LINE_NAME', 'LN_IC50', 'AUC', 'RMSE', 'Z_SCORE', 'SANGER_MODEL_ID', 'TCGA_DESC', 'DATASET', 'MAX_CONC', 'MIN_CONC', 'SCREENING_SITE', 'SCREENING_MEDIUM', 'Screen Medium', 'CELL_ID', 'CELL_LINE_NAME_ANN', 'TISSUE_ANN', 'CANCER_TYPE_ANN', 'MOA_LABEL_FINAL', 'TARGET_GENES_FINAL', 'PATHWAY_NAME_FINAL', 'MOA_SOURCE_FINAL', 'DOMAIN_COMBINED', 'pubchem_match_name', 'pubchem_cid', 'pubchem_canonical_smiles', 'pubchem_isomeric_smiles', 'pubchem_inchi', 'pubchem_inchikey', 'chembl_match_name', 'chembl_id', 'chembl_pref_name', 'chembl_smiles', 'chembl_inchi', 'chembl_inchikey', 'smiles_best', 'inchi_best', 'inchikey_best', 'source_best', 'pubchem_chembl_inchikey_disagree', 'lookup_candidates_used', 'n_lookup_candidates', 'lookup_error', 'smiles_best_v2', 'inchi_best_v2', 'inchikey_best_v2', 'source_best_v2', 'target_genes_in_expr_str', 'ppi_neighbor_genes_in_expr_str', 'MOA_LABEL_COLLAPSED', 'MOA_LABEL_COLLAPSED_TRAIN'}
    gene_pattern = '^[A-Za-z0-9\\-\\.]+$'
    return [c for c in df.columns if c not in non_expr_known and pd.api.types.is_numeric_dtype(df[c]) and bool(pd.Series([c]).astype(str).str.match(gene_pattern).iloc[0]) and (not str(c).startswith('PC')) and (not str(c).startswith('SVD'))]

def prepare_internal_tables(all_df: pd.DataFrame, cfg: ConfigDruXRV5):
    df = all_df.copy()
    df.columns = [str(c).replace('\n', ' ').strip() for c in df.columns]
    if 'CELL_LINE_NAME' not in df.columns:
        alt = pick_first_available(df, ['CELL_LINE_NAME_ANN', 'Sample Name'])
        if alt is None:
            raise ValueError('Input must contain CELL_LINE_NAME or CELL_LINE_NAME_ANN')
        df = df.rename(columns={alt: 'CELL_LINE_NAME'})
    if 'TCGA_DESC' not in df.columns:
        alt = pick_first_available(df, ['TISSUE_ANN', 'CANCER_TYPE_ANN', 'Tissue', 'Histology'])
        df['TCGA_DESC'] = df[alt] if alt is not None else 'UNCLASSIFIED'
    if 'AUC' not in df.columns or 'DRUG_ID' not in df.columns:
        raise ValueError('Input must contain AUC and DRUG_ID')
    if 'LN_IC50' not in df.columns:
        raise ValueError('Dual-response mode requires an LN_IC50 column. Use the AUC-only strict script if LN_IC50 is unavailable.')
    df['DOMAIN_COMBINED'] = build_domain_column(df)
    expr_cols = infer_expression_columns(df)
    if not expr_cols:
        raise ValueError('Could not infer expression columns')
    expr_graph_df = df[['CELL_LINE_NAME'] + expr_cols].drop_duplicates(subset=['CELL_LINE_NAME']).reset_index(drop=True)
    pairs_df = df.drop(columns=expr_cols, errors='ignore').copy()
    drug_exclude_cols = set(expr_cols) | {'COSMIC_ID', 'CELL_LINE_NAME', 'CELL_LINE_NAME_ANN', 'TISSUE_ANN', 'CANCER_TYPE_ANN', 'TCGA_DESC', 'AUC', 'LN_IC50', 'RMSE', 'Z_SCORE', 'SANGER_MODEL_ID', 'DATASET', 'MAX_CONC', 'MIN_CONC', 'SCREENING_SITE', 'SCREENING_MEDIUM', 'Screen Medium', 'CELL_ID', 'DOMAIN_COMBINED'}
    drug_cols = [c for c in df.columns if c not in drug_exclude_cols]
    drug_raw = df[drug_cols].copy()
    agg_map = {}
    for c in drug_raw.columns:
        if c == 'DRUG_ID':
            continue
        agg_map[c] = first_nonnull_value if pd.api.types.is_numeric_dtype(drug_raw[c]) else first_nonempty_value
    drug_df = drug_raw.groupby('DRUG_ID', as_index=False).agg(agg_map)
    smiles_col = choose_smiles_column(drug_df)
    if smiles_col is None:
        raise ValueError('No SMILES column found')
    valid_mask = drug_df[smiles_col].map(is_valid_smiles_value).astype(bool)
    invalid_ids = drug_df.loc[~valid_mask, 'DRUG_ID'].astype(int).tolist()
    if cfg.valid_smiles_only and invalid_ids:
        print(f'[SMILES] Excluding {len(invalid_ids)} drugs without valid RDKit-parseable SMILES.')
    drug_graph_df = drug_df.loc[valid_mask].copy().reset_index(drop=True)
    graph_drug_ids = set(drug_graph_df['DRUG_ID'].astype(int))
    graph_pairs_df = pairs_df[pairs_df['DRUG_ID'].astype(int).isin(graph_drug_ids)].reset_index(drop=True)
    return (graph_pairs_df, drug_graph_df, expr_graph_df, expr_cols)

def load_input_data(cfg: ConfigDruXRV5, all_df: Optional[pd.DataFrame]=None):
    if all_df is None or not isinstance(all_df, pd.DataFrame):
        raise ValueError('Pass the input dataframe explicitly, e.g. run_druxr_v5(CFG_DRUXR_V5, all_df=all_merged_full).')
    print('Loaded unified input from explicit function argument: all_df')
    return prepare_internal_tables(all_df, cfg)

def split_by_drug(pairs_df: pd.DataFrame, train_frac: float, val_frac: float, test_frac: float, seed: int):
    assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-08
    drug_ids = np.array(sorted(pd.unique(pairs_df['DRUG_ID'].astype(int))))
    if len(drug_ids) < 3:
        raise ValueError('Need at least 3 unique drugs for unseen-drug split')
    tr_ids, tmp_ids = train_test_split(drug_ids, test_size=1.0 - train_frac, random_state=seed)
    rel_test = test_frac / (val_frac + test_frac)
    va_ids, te_ids = train_test_split(tmp_ids, test_size=rel_test, random_state=seed)
    tr = pairs_df[pairs_df['DRUG_ID'].astype(int).isin(set(tr_ids))].copy().reset_index(drop=True)
    va = pairs_df[pairs_df['DRUG_ID'].astype(int).isin(set(va_ids))].copy().reset_index(drop=True)
    te = pairs_df[pairs_df['DRUG_ID'].astype(int).isin(set(te_ids))].copy().reset_index(drop=True)
    return (tr, va, te, sorted(map(int, tr_ids)), sorted(map(int, va_ids)), sorted(map(int, te_ids)))

def fit_transform_expression_train_only(expr_graph_df: pd.DataFrame, expr_cols: List[str], train_pairs_df: pd.DataFrame, cfg: ConfigDruXRV5):
    expr_graph_df = expr_graph_df.drop_duplicates(subset=['CELL_LINE_NAME']).copy()
    expr_graph_df = expr_graph_df.set_index('CELL_LINE_NAME')
    expr_values_all = expr_graph_df[expr_cols].astype(np.float32).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    train_cells = sorted(set(train_pairs_df['CELL_LINE_NAME'].astype(str)) & set(expr_values_all.index.astype(str)))
    if len(train_cells) < 3:
        raise ValueError('Too few train cells for train-only expression PCA')
    scaler = StandardScaler()
    x_train = scaler.fit_transform(expr_values_all.loc[train_cells].values)
    x_all = scaler.transform(expr_values_all.values)
    pca_dim = int(min(cfg.expr_pca_dim, x_train.shape[0] - 1, x_train.shape[1]))
    pca = PCA(n_components=pca_dim, random_state=cfg.seed)
    pca.fit(x_train)
    emb_all = pca.transform(x_all).astype(np.float32)
    expr_embed_df = pd.DataFrame(emb_all, index=expr_values_all.index, columns=[f'PC{i + 1}' for i in range(pca_dim)])
    expr_raw_df = expr_values_all.astype(np.float32)
    expr_gene_set = set(expr_cols)
    print(f'[strict_v5] Expression PCA shape, train-only fit: {expr_embed_df.shape}')
    return (scaler, pca, expr_embed_df, expr_raw_df, expr_gene_set)

def _is_pathway_leakage_col(c: str) -> bool:
    s = str(c)
    return s.startswith('PATH_') or s.startswith('PATHWAY_NAME_FINAL_DUMMY') or s.startswith('PATHWAY_NAME_FINAL_') or ('PATHWAY_NAME_FINAL_DUMMY' in s)

def fit_transform_drug_features_train_only(drug_df: pd.DataFrame, train_drug_ids: List[int], cfg: ConfigDruXRV5):
    df = drug_df.copy()
    if 'DRUG_ID' not in df.columns:
        raise ValueError('drug_df must contain DRUG_ID')
    numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns.tolist() if c != 'DRUG_ID']
    if cfg.remove_pathway_feature_leakage:
        leak_cols = [c for c in numeric_cols if _is_pathway_leakage_col(c)]
        if leak_cols:
            numeric_cols = [c for c in numeric_cols if c not in leak_cols]
            print(f'[strict_v5] Removed {len(leak_cols)} pathway-leakage numeric feature columns.')
    if not numeric_cols:
        raise ValueError('No numeric drug features found after leakage filtering')
    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    train_mask = df['DRUG_ID'].astype(int).isin(set(map(int, train_drug_ids)))
    train_part = df.loc[train_mask, numeric_cols]
    all_nan_train = [c for c in numeric_cols if train_part[c].notna().sum() == 0]
    if all_nan_train:
        print(f'[WARN] Dropping all-NaN train drug feature columns: {all_nan_train}')
        numeric_cols = [c for c in numeric_cols if c not in all_nan_train]
        train_part = train_part[numeric_cols]
    med = train_part.median(numeric_only=True)
    df[numeric_cols] = df[numeric_cols].fillna(med).fillna(0.0)
    if cfg.scale_drug_features:
        means = df.loc[train_mask, numeric_cols].mean(axis=0)
        stds = df.loc[train_mask, numeric_cols].std(axis=0, ddof=0).replace(0.0, 1.0).fillna(1.0)
        df[numeric_cols] = (df[numeric_cols] - means) / stds
        df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], 0.0).fillna(0.0)
    drug_feat_df = df.set_index('DRUG_ID')
    return (drug_feat_df, numeric_cols)

def build_moa_targets_train_vocab(drug_df_all: pd.DataFrame, train_drug_ids: List[int], min_moa_count: int):
    moa_col = None
    for c in ['MOA_LABEL_COLLAPSED_TRAIN', 'MOA_LABEL_COLLAPSED', 'MOA_LABEL_FINAL']:
        if c in drug_df_all.columns:
            moa_col = c
            break
    if moa_col is None:
        return (pd.DataFrame(index=drug_df_all['DRUG_ID'].values), [])
    train_ids = set(map(int, train_drug_ids))
    counts: Dict[str, int] = {}
    for x in drug_df_all.loc[drug_df_all['DRUG_ID'].astype(int).isin(train_ids), moa_col].dropna():
        for lab in safe_split_pipe(x):
            counts[lab] = counts.get(lab, 0) + 1
    vocab = sorted([k for k, v in counts.items() if v >= min_moa_count])
    if not vocab:
        return (pd.DataFrame(index=drug_df_all['DRUG_ID'].values), [])
    rows = []
    for _, r in drug_df_all[['DRUG_ID', moa_col]].iterrows():
        labs = set([x for x in safe_split_pipe(r[moa_col]) if x in vocab])
        rows.append([1.0 if m in labs else 0.0 for m in vocab])
    return (pd.DataFrame(rows, index=drug_df_all['DRUG_ID'].values, columns=vocab, dtype=np.float32), vocab)

def build_pathway_targets_train_vocab(drug_df_all: pd.DataFrame, train_drug_ids: List[int], min_pathway_count: int):
    pathway_col = None
    for c in ['PATHWAY_NAME_FINAL', 'PATHWAY_NAME']:
        if c in drug_df_all.columns:
            pathway_col = c
            break
    if pathway_col is None:
        return (pd.DataFrame(index=drug_df_all['DRUG_ID'].values), [])
    train_ids = set(map(int, train_drug_ids))
    train_values = drug_df_all.loc[drug_df_all['DRUG_ID'].astype(int).isin(train_ids), pathway_col].fillna('UNKNOWN_PATHWAY').astype(str)
    counts = train_values.value_counts()
    vocab = sorted([p for p, n in counts.items() if n >= min_pathway_count])
    if not vocab:
        return (pd.DataFrame(index=drug_df_all['DRUG_ID'].values), [])
    rows = []
    for _, r in drug_df_all[['DRUG_ID', pathway_col]].iterrows():
        p = str(r[pathway_col]) if pd.notna(r[pathway_col]) else 'UNKNOWN_PATHWAY'
        rows.append([1.0 if p == v else 0.0 for v in vocab])
    return (pd.DataFrame(rows, index=drug_df_all['DRUG_ID'].values, columns=vocab, dtype=np.float32), vocab)

def derive_selectivity_labels(pairs_df: pd.DataFrame, cfg: ConfigDruXRV5):
    subtype_counts = pairs_df['TCGA_DESC'].value_counts(dropna=False)
    usable_subtypes = [s for s in subtype_counts.index if pd.notna(s) and s != 'UNCLASSIFIED' and (subtype_counts[s] >= cfg.min_subtype_size)]
    df = pairs_df[pairs_df['TCGA_DESC'].isin(usable_subtypes)].copy()
    all_drugs = sorted(pd.unique(pairs_df['DRUG_ID'].astype(int)))
    if df.empty:
        return (pd.DataFrame(index=all_drugs), [], pd.DataFrame(columns=['DRUG_ID', 'TCGA_DESC', 'selectivity_advantage']))
    drug_sub_mean = df.groupby(['DRUG_ID', 'TCGA_DESC'], as_index=False)['AUC'].mean().rename(columns={'AUC': 'auc_subtype'})
    drug_sub_n = df.groupby(['DRUG_ID', 'TCGA_DESC'], as_index=False)['CELL_LINE_NAME'].nunique().rename(columns={'CELL_LINE_NAME': 'n_cell_lines_subtype'})
    drug_global_mean = df.groupby('DRUG_ID', as_index=False)['AUC'].mean().rename(columns={'AUC': 'auc_global'})
    score_df = drug_sub_mean.merge(drug_global_mean, on='DRUG_ID', how='left').merge(drug_sub_n, on=['DRUG_ID', 'TCGA_DESC'], how='left')
    score_df['selectivity_advantage'] = (score_df['auc_global'] - score_df['auc_subtype']).fillna(0.0)
    score_df['selectivity_adv_pos'] = score_df['selectivity_advantage'].clip(lower=0.0)
    score_df['selectivity_strength'] = score_df.groupby('DRUG_ID')['selectivity_adv_pos'].transform(lambda s: minmax01(s).astype(np.float32)).astype(np.float32)
    score_df['selectivity_rank_within_drug'] = score_df.groupby('DRUG_ID')['selectivity_advantage'].transform(lambda s: zscore_series(s).astype(np.float32)).astype(np.float32)
    pos_counts = score_df[score_df['selectivity_advantage'] >= cfg.selectivity_effect_threshold].groupby('TCGA_DESC')['DRUG_ID'].nunique()
    refined_subtypes = [s for s in usable_subtypes if pos_counts.get(s, 0) >= cfg.selectivity_min_positive_drugs]
    if not refined_subtypes:
        refined_subtypes = usable_subtypes
    label_rows = []
    for drug_id in all_drugs:
        sub = score_df[(score_df['DRUG_ID'].astype(int) == int(drug_id)) & score_df['TCGA_DESC'].isin(refined_subtypes)].copy()
        sub = sub.sort_values(['selectivity_advantage', 'n_cell_lines_subtype'], ascending=[False, False])
        keep = sub[sub['selectivity_advantage'] >= cfg.selectivity_effect_threshold].head(cfg.max_selective_subtypes_per_drug)
        keep_labels = set(keep['TCGA_DESC'].tolist())
        label_rows.append([1.0 if s in keep_labels else 0.0 for s in refined_subtypes])
    sel_df = pd.DataFrame(label_rows, index=all_drugs, columns=refined_subtypes, dtype=np.float32)
    score_df = score_df[score_df['TCGA_DESC'].isin(refined_subtypes)].reset_index(drop=True)
    return (sel_df, refined_subtypes, score_df)

def add_pair_level_rank_targets_train_only(train_pairs_df: pd.DataFrame, score_df: Optional[pd.DataFrame], cfg: ConfigDruXRV5):
    df = train_pairs_df.copy()
    if score_df is not None and len(score_df) > 0:
        merge_cols = [c for c in ['DRUG_ID', 'TCGA_DESC', 'selectivity_advantage', 'selectivity_strength', 'selectivity_rank_within_drug'] if c in score_df.columns]
        df = df.merge(score_df[merge_cols], on=['DRUG_ID', 'TCGA_DESC'], how='left')
    else:
        df['selectivity_advantage'] = 0.0
    for c in ['selectivity_advantage', 'selectivity_strength', 'selectivity_rank_within_drug']:
        if c not in df.columns:
            df[c] = 0.0
        df[c] = df[c].fillna(0.0).astype(np.float32)
    df['sensitivity_proxy_global'] = -zscore_series(df['AUC']).astype(np.float32)
    df['sensitivity_proxy_subtype'] = df.groupby('TCGA_DESC')['AUC'].transform(lambda s: (-zscore_series(s)).astype(np.float32)).astype(np.float32)
    df['subtype_specificity_proxy'] = df.groupby('DRUG_ID')['selectivity_advantage'].transform(lambda s: zscore_series(s).astype(np.float32)).astype(np.float32)
    clip = float(cfg.subtype_proxy_clip)
    sens_global = np.clip(df['sensitivity_proxy_global'], -2.5, 2.5) / 2.5
    sens_subtype = np.clip(df['sensitivity_proxy_subtype'], -2.5, 2.5) / 2.5
    subtype_component = np.clip(df['subtype_specificity_proxy'], -clip, clip) / clip
    sel_component = np.clip(df['selectivity_rank_within_drug'], -clip, clip) / clip
    sel_strength = np.clip(df['selectivity_strength'], 0.0, 1.0)
    if cfg.rank_within_subtype:
        rank_raw = 0.4 * sens_subtype + 0.25 * sens_global + 0.2 * subtype_component + 0.1 * sel_component + 0.05 * sel_strength
        df['rank_target'] = rank_raw.astype(np.float32)
        df['rank_target'] = df.groupby('TCGA_DESC')['rank_target'].transform(lambda s: minmax01(s).astype(np.float32)).astype(np.float32)
    else:
        rank_raw = cfg.rank_global_mix * (0.55 * sens_global + 0.45 * sel_strength) + (1.0 - cfg.rank_global_mix) * (0.5 * sens_subtype + 0.3 * subtype_component + 0.2 * sel_component)
        df['rank_target'] = minmax01(rank_raw).astype(np.float32)
    df['rank_group'] = df['TCGA_DESC'].fillna('__UNK__').astype(str)
    df['rank_mask'] = 1.0
    return df

def prepare_split_auxiliary_targets(tr_df: pd.DataFrame, va_df: pd.DataFrame, te_df: pd.DataFrame, drug_df: pd.DataFrame, train_drug_ids: List[int], cfg: ConfigDruXRV5):
    sel_train_df, subtype_vocab, sel_score_train = derive_selectivity_labels(tr_df, cfg)
    all_drug_ids = sorted(pd.unique(pd.concat([tr_df, va_df, te_df], axis=0)['DRUG_ID'].astype(int)))
    sel_targets_df = pd.DataFrame(0.0, index=all_drug_ids, columns=subtype_vocab, dtype=np.float32)
    if len(subtype_vocab) and (not sel_train_df.empty):
        common = [i for i in sel_train_df.index if i in sel_targets_df.index]
        sel_targets_df.loc[common, subtype_vocab] = sel_train_df.loc[common, subtype_vocab].values
    sel_train_valid_ids = set(map(int, sel_train_df.index)) if len(subtype_vocab) else set()
    tr_df = add_pair_level_rank_targets_train_only(tr_df, sel_score_train, cfg)
    for split_df in [va_df, te_df]:
        split_df['rank_target'] = 0.0
        split_df['rank_group'] = split_df['TCGA_DESC'].fillna('__UNK__').astype(str)
        split_df['rank_mask'] = 0.0
    moa_targets_df, moa_vocab = build_moa_targets_train_vocab(drug_df, train_drug_ids, cfg.min_moa_count)
    pathway_targets_df, pathway_vocab = build_pathway_targets_train_vocab(drug_df, train_drug_ids, cfg.min_pathway_count)
    return (tr_df, va_df, te_df, moa_targets_df, moa_vocab, pathway_targets_df, pathway_vocab, sel_targets_df, subtype_vocab, sel_train_valid_ids)

def compute_tic_feature_vector(drug_row: pd.Series, expr_row: pd.Series, expr_columns_set: set, cfg: ConfigDruXRV5):
    target_genes, neighbor_genes = ([], [])
    for c in ['target_genes_in_expr_str', 'TARGET_GENES_FINAL']:
        if c in drug_row.index:
            target_genes = parse_gene_list(drug_row[c])
            if target_genes:
                break
    for c in ['ppi_neighbor_genes_in_expr_str']:
        if c in drug_row.index:
            neighbor_genes = parse_gene_list(drug_row[c])
            if neighbor_genes:
                break
    target_genes = [g for g in target_genes if g in expr_columns_set]
    neighbor_genes = [g for g in neighbor_genes if g in expr_columns_set]
    target_vals = expr_row[target_genes].values.astype(np.float32) if target_genes else np.array([], dtype=np.float32)
    neighbor_vals = expr_row[neighbor_genes].values.astype(np.float32) if neighbor_genes else np.array([], dtype=np.float32)
    target_vals = np.nan_to_num(target_vals, nan=0.0, posinf=0.0, neginf=0.0)
    neighbor_vals = np.nan_to_num(neighbor_vals, nan=0.0, posinf=0.0, neginf=0.0)
    n_target, n_neighbor = (len(target_vals), len(neighbor_vals))
    tstats, nstats = (_expr_stats(target_vals), _expr_stats(neighbor_vals))
    pooled = np.concatenate([target_vals, neighbor_vals]) if n_target + n_neighbor else np.array([], dtype=np.float32)
    pooled_std = float(pooled.std(ddof=0)) if pooled.size else 0.0
    tic_valid = float(n_target >= cfg.tic_min_target_genes and n_neighbor >= cfg.tic_min_neighbor_genes)
    tic_target = float((tstats['mean'] - nstats['mean']) / (pooled_std if pooled_std > 1e-06 else 1.0)) if tic_valid else 0.0

    def get_num(col: str) -> float:
        if col not in drug_row.index:
            return 0.0
        v = pd.to_numeric(pd.Series([drug_row[col]]), errors='coerce').iloc[0]
        return 0.0 if pd.isna(v) or np.isinf(v) else float(v)
    ratio = float(tstats['mean'] / (abs(nstats['mean']) + 0.001)) if n_neighbor else 0.0
    feat = np.array([float(n_target), float(n_neighbor), float(n_target) / float(max(len(target_genes), 1)) if target_genes else 0.0, float(n_neighbor) / float(max(len(neighbor_genes), 1)) if neighbor_genes else 0.0, tstats['mean'], nstats['mean'], tstats['mean'] - nstats['mean'], abs(tstats['mean'] - nstats['mean']), tstats['std'], nstats['std'], tstats['q25'], tstats['q75'], nstats['q25'], nstats['q75'], tstats['max'], nstats['max'], tstats['min'], nstats['min'], tstats['frac_high'], nstats['frac_high'], ratio, get_num('ppi_target_degree_mean'), get_num('ppi_num_neighbors_in_expr'), tic_valid], dtype=np.float32)
    feat = np.nan_to_num(feat, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return (feat, tic_target, tic_valid)

class GraphPairDatasetV2Strict(Dataset):

    def __init__(self, pairs_df: pd.DataFrame, drug_df_full: pd.DataFrame, drug_feat_df: pd.DataFrame, drug_feature_cols: List[str], expr_embed_df: pd.DataFrame, expr_raw_df: pd.DataFrame, expr_columns_set: set, drug_graph_dict: Dict[int, Data], moa_targets_df: pd.DataFrame, selectivity_targets_df: pd.DataFrame, pathway_targets_df: pd.DataFrame, domain_to_id: Dict[str, int], rank_group_to_id: Dict[str, int], selectivity_train_valid_ids: set, cfg: ConfigDruXRV5):
        self.df = pairs_df.reset_index(drop=True).copy()
        self.drug_df_full = drug_df_full.set_index('DRUG_ID')
        self.drug_feat_df = drug_feat_df
        self.drug_feature_cols = drug_feature_cols
        self.expr_embed_df = expr_embed_df
        self.expr_raw_df = expr_raw_df
        self.expr_columns_set = expr_columns_set
        self.drug_graph_dict = drug_graph_dict
        self.moa_targets_df = moa_targets_df
        self.selectivity_targets_df = selectivity_targets_df
        self.pathway_targets_df = pathway_targets_df
        self.domain_to_id = domain_to_id
        self.rank_group_to_id = rank_group_to_id
        self.selectivity_train_valid_ids = set(map(int, selectivity_train_valid_ids))
        self.cfg = cfg

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        drug_id = int(row['DRUG_ID'])
        cell_line = row['CELL_LINE_NAME']
        domain = str(row['DOMAIN_COMBINED'])
        rank_group = str(row.get('rank_group', row.get('TCGA_DESC', '__UNK__')))
        graph = self.drug_graph_dict[drug_id]
        drug_row_full = self.drug_df_full.loc[drug_id]
        drug_name = str(drug_id)
        for _name_col in ['DRUG_NAME', 'COMPOUND_NAME', 'chembl_pref_name', 'pubchem_match_name']:
            if _name_col in drug_row_full.index and pd.notna(drug_row_full[_name_col]) and str(drug_row_full[_name_col]).strip():
                drug_name = str(drug_row_full[_name_col]).strip()
                break
        drug_tab = self.drug_feat_df.loc[drug_id, self.drug_feature_cols].values.astype(np.float32)
        cell_expr = self.expr_embed_df.loc[cell_line].values.astype(np.float32)
        expr_row = self.expr_raw_df.loc[cell_line]
        tic_feat, tic_target, tic_valid = compute_tic_feature_vector(drug_row_full, expr_row, self.expr_columns_set, self.cfg)
        moa_target = self.moa_targets_df.loc[drug_id].values.astype(np.float32) if drug_id in self.moa_targets_df.index and len(self.moa_targets_df.columns) else np.zeros((0,), dtype=np.float32)
        sel_target = self.selectivity_targets_df.loc[drug_id].values.astype(np.float32) if drug_id in self.selectivity_targets_df.index and len(self.selectivity_targets_df.columns) else np.zeros((0,), dtype=np.float32)
        pathway_target = self.pathway_targets_df.loc[drug_id].values.astype(np.float32) if drug_id in self.pathway_targets_df.index and len(self.pathway_targets_df.columns) else np.zeros((0,), dtype=np.float32)
        sel_mask = float(drug_id in self.selectivity_train_valid_ids and len(sel_target) > 0)
        ln_ic50_value = pd.to_numeric(row.get('LN_IC50', np.nan), errors='coerce')
        ln_ic50_valid = float(pd.notna(ln_ic50_value) and np.isfinite(float(ln_ic50_value)))
        ln_ic50_target = float(ln_ic50_value) if ln_ic50_valid else 0.0
        return {'graph': graph, 'drug_id': drug_id, 'drug_name': drug_name, 'cell_line': cell_line, 'tcga_desc': str(row.get('TCGA_DESC', '')), 'drug_tab': torch.tensor(drug_tab, dtype=torch.float32), 'cell_expr': torch.tensor(cell_expr, dtype=torch.float32), 'tic_feat': torch.tensor(tic_feat, dtype=torch.float32), 'response_target': torch.tensor([float(row['AUC'])], dtype=torch.float32), 'ln_ic50_target': torch.tensor([float(ln_ic50_target)], dtype=torch.float32), 'ln_ic50_mask': torch.tensor([float(ln_ic50_valid)], dtype=torch.float32), 'tic_target': torch.tensor([float(tic_target)], dtype=torch.float32), 'rank_target': torch.tensor([float(row.get('rank_target', 0.0))], dtype=torch.float32), 'rank_mask': torch.tensor([float(row.get('rank_mask', 0.0))], dtype=torch.float32), 'moa_target': torch.tensor(moa_target, dtype=torch.float32), 'sel_target': torch.tensor(sel_target, dtype=torch.float32), 'pathway_target': torch.tensor(pathway_target, dtype=torch.float32), 'moa_mask': torch.tensor([float(len(moa_target) > 0)], dtype=torch.float32), 'sel_mask': torch.tensor([sel_mask], dtype=torch.float32), 'pathway_mask': torch.tensor([float(len(pathway_target) > 0)], dtype=torch.float32), 'tic_mask': torch.tensor([float(tic_valid)], dtype=torch.float32), 'domain_target': torch.tensor(self.domain_to_id.get(domain, 0), dtype=torch.long), 'rank_group': torch.tensor(self.rank_group_to_id.get(rank_group, 0), dtype=torch.long)}

def collate_graph_batch_v2_strict(batch: List[Dict[str, Any]]) -> Dict[str, Any]:

    def stack_optional(key: str):
        return torch.stack([item[key] for item in batch], dim=0) if len(batch[0][key]) > 0 else torch.empty((len(batch), 0))
    return {'graph_batch': Batch.from_data_list([item['graph'] for item in batch]), 'drug_tab': torch.stack([item['drug_tab'] for item in batch], dim=0), 'cell_expr': torch.stack([item['cell_expr'] for item in batch], dim=0), 'tic_feat': torch.stack([item['tic_feat'] for item in batch], dim=0), 'response_target': torch.stack([item['response_target'] for item in batch], dim=0), 'ln_ic50_target': torch.stack([item['ln_ic50_target'] for item in batch], dim=0), 'ln_ic50_mask': torch.stack([item['ln_ic50_mask'] for item in batch], dim=0), 'tic_target': torch.stack([item['tic_target'] for item in batch], dim=0), 'rank_target': torch.stack([item['rank_target'] for item in batch], dim=0), 'rank_mask': torch.stack([item['rank_mask'] for item in batch], dim=0), 'moa_target': stack_optional('moa_target'), 'sel_target': stack_optional('sel_target'), 'pathway_target': stack_optional('pathway_target'), 'moa_mask': torch.stack([item['moa_mask'] for item in batch], dim=0), 'sel_mask': torch.stack([item['sel_mask'] for item in batch], dim=0), 'pathway_mask': torch.stack([item['pathway_mask'] for item in batch], dim=0), 'tic_mask': torch.stack([item['tic_mask'] for item in batch], dim=0), 'domain_target': torch.stack([item['domain_target'] for item in batch], dim=0), 'rank_group': torch.stack([item['rank_group'] for item in batch], dim=0), 'drug_id': torch.tensor([item['drug_id'] for item in batch], dtype=torch.long), 'drug_name': [item.get('drug_name', str(item['drug_id'])) for item in batch], 'cell_line': [item['cell_line'] for item in batch], 'tcga_desc': [item['tcga_desc'] for item in batch]}

class GradientReversalFn(torch.autograd.Function):

    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return (-ctx.lambd * grad_output, None)

class GRL(nn.Module):

    def __init__(self, lambd=1.0):
        super().__init__()
        self.lambd = lambd

    def forward(self, x):
        return GradientReversalFn.apply(x, self.lambd)

class MolecularGIN(nn.Module):

    def __init__(self, node_dim=7, edge_dim=4, hidden_dim=64, dropout=0.35):
        super().__init__()
        self.node_proj = nn.Linear(node_dim, hidden_dim)
        self.edge_proj = nn.Linear(edge_dim, hidden_dim)
        nn1 = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
        nn2 = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
        self.conv1 = GINEConv(nn1, edge_dim=hidden_dim)
        self.conv2 = GINEConv(nn2, edge_dim=hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, batch_graph):
        x = self.node_proj(batch_graph.x)
        edge_attr = self.edge_proj(batch_graph.edge_attr) if batch_graph.edge_attr.numel() > 0 else batch_graph.edge_attr
        x = F.relu(self.conv1(x, batch_graph.edge_index, edge_attr))
        x = self.dropout(x)
        x = F.relu(self.conv2(x, batch_graph.edge_index, edge_attr))
        return global_add_pool(x, batch_graph.batch)

class MLPEncoder(nn.Module):

    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, out_dim), nn.ReLU())

    def forward(self, x):
        return self.net(x)

class DruXRGraphV2Strict(nn.Module):

    def __init__(self, drug_tab_dim: int, cell_dim: int, tic_dim: int, n_moa: int, n_sel: int, n_pathway: int, n_domains: int, enable_domain_adv: bool, cfg: ConfigDruXRV5):
        super().__init__()
        _ensure_v5_config_defaults(cfg)
        tic_dim = int(getattr(cfg, 'tic_feature_dim', tic_dim))
        self.enable_domain_adv = bool(enable_domain_adv and n_domains > 1)
        self.graph_encoder = MolecularGIN(7, 4, cfg.graph_hidden_dim, cfg.dropout)
        self.drug_tab_encoder = MLPEncoder(drug_tab_dim, cfg.tab_hidden_dim, cfg.tab_hidden_dim, cfg.dropout)
        self.cell_encoder = MLPEncoder(cell_dim, cfg.cell_hidden_dim, cfg.cell_hidden_dim, cfg.dropout)
        self.tic_encoder = MLPEncoder(tic_dim, cfg.tic_hidden_dim, cfg.tic_hidden_dim, cfg.dropout)
        drug_context_dim = cfg.graph_hidden_dim + cfg.tab_hidden_dim + cfg.tic_hidden_dim
        self.cell_gate = nn.Sequential(nn.Linear(drug_context_dim, cfg.cell_hidden_dim), nn.Sigmoid())
        self.cell_context_bias = nn.Sequential(nn.Linear(drug_context_dim, cfg.cell_hidden_dim), nn.Tanh())
        fusion_in = cfg.graph_hidden_dim + cfg.tab_hidden_dim + cfg.cell_hidden_dim + cfg.tic_hidden_dim
        self.fusion = nn.Sequential(nn.Linear(fusion_in, cfg.fusion_hidden_dim), nn.ReLU(), nn.Dropout(cfg.dropout), nn.Linear(cfg.fusion_hidden_dim, cfg.fusion_hidden_dim), nn.ReLU())
        self.response_head = nn.Linear(cfg.fusion_hidden_dim, 1)
        self.ln_ic50_head = nn.Linear(cfg.fusion_hidden_dim, 1)
        self.tic_head = nn.Linear(cfg.fusion_hidden_dim, 1)
        self.rank_head = nn.Linear(cfg.fusion_hidden_dim, 1)
        self.moa_head = nn.Linear(cfg.fusion_hidden_dim, n_moa) if n_moa > 0 else None
        self.sel_head = nn.Linear(cfg.fusion_hidden_dim, n_sel) if n_sel > 0 else None
        self.pathway_head = nn.Linear(cfg.fusion_hidden_dim, n_pathway) if n_pathway > 0 else None
        if self.enable_domain_adv:
            self.grl = GRL(1.0)
            self.domain_head = nn.Sequential(nn.Linear(cfg.fusion_hidden_dim, 128), nn.ReLU(), nn.Linear(128, n_domains))
        else:
            self.grl = None
            self.domain_head = None

    def encode_shared(self, graph_batch, drug_tab, cell_expr, tic_feat):
        g = self.graph_encoder(graph_batch)
        d = self.drug_tab_encoder(drug_tab)
        c = self.cell_encoder(cell_expr)
        t = self.tic_encoder(tic_feat)
        drug_context = torch.cat([g, d, t], dim=1)
        gate = self.cell_gate(drug_context)
        bias = self.cell_context_bias(drug_context)
        c = c * (0.5 + gate) + 0.1 * bias
        return self.fusion(torch.cat([g, d, c, t], dim=1))

    def forward(self, batch):
        z = self.encode_shared(batch['graph_batch'], batch['drug_tab'], batch['cell_expr'], batch['tic_feat'])
        out = {'shared': z, 'response': self.response_head(z), 'ln_ic50': self.ln_ic50_head(z), 'tic': self.tic_head(z), 'rank': self.rank_head(z)}
        if self.moa_head is not None:
            out['moa'] = self.moa_head(z)
        if self.sel_head is not None:
            out['selectivity'] = self.sel_head(z)
        if self.pathway_head is not None:
            out['pathway'] = self.pathway_head(z)
        if self.enable_domain_adv:
            out['domain'] = self.domain_head(self.grl(z))
        return out

def move_batch_to_device(batch: Dict[str, Any], device: str) -> Dict[str, Any]:
    out = {}
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            out[k] = v.to(device)
        elif hasattr(v, 'to'):
            out[k] = v.to(device)
        else:
            out[k] = v
    return out

def masked_bce_with_logits(logits: torch.Tensor, targets: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    if logits.numel() == 0 or targets.numel() == 0:
        return torch.tensor(0.0, device=mask.device)
    loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    loss = loss.mean(dim=1, keepdim=True) * mask
    return loss.sum() / mask.sum().clamp_min(1.0)

def masked_smooth_l1(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor, beta: float=1.0) -> torch.Tensor:
    loss = F.smooth_l1_loss(pred, target, reduction='none', beta=beta) * mask
    return loss.sum() / mask.sum().clamp_min(1.0)

def pairwise_rank_loss(scores: torch.Tensor, targets: torch.Tensor, groups: torch.Tensor, mask: Optional[torch.Tensor]=None) -> torch.Tensor:
    scores, targets, groups = (scores.view(-1), targets.view(-1), groups.view(-1))
    if mask is not None:
        keep = mask.view(-1) > 0
        scores, targets, groups = (scores[keep], targets[keep], groups[keep])
    if scores.numel() < 2:
        return torch.tensor(0.0, device=scores.device)
    losses = []
    for g in torch.unique(groups):
        idx = torch.where(groups == g)[0]
        if idx.numel() < 2:
            continue
        s, y = (scores[idx], targets[idx])
        diff_y = y.unsqueeze(1) - y.unsqueeze(0)
        diff_s = s.unsqueeze(1) - s.unsqueeze(0)
        sign = torch.sign(diff_y)
        valid = sign != 0
        if valid.sum() > 0:
            losses.append(F.softplus(-sign[valid] * diff_s[valid]).mean())
    return torch.stack(losses).mean() if losses else torch.tensor(0.0, device=scores.device)

def compute_multilabel_metrics(y_true: np.ndarray, y_score: np.ndarray) -> Dict[str, float]:
    out = {'auroc_macro': np.nan, 'auprc_macro': np.nan}
    if y_true.size == 0 or y_score.size == 0:
        return out
    aurocs, auprcs = ([], [])
    for j in range(y_true.shape[1]):
        yt, ys = (y_true[:, j], y_score[:, j])
        if len(np.unique(yt)) < 2:
            continue
        try:
            aurocs.append(roc_auc_score(yt, ys))
        except Exception:
            pass
        try:
            auprcs.append(average_precision_score(yt, ys))
        except Exception:
            pass
    if aurocs:
        out['auroc_macro'] = float(np.mean(aurocs))
    if auprcs:
        out['auprc_macro'] = float(np.mean(auprcs))
    return out

def response_metrics_from_df(pred_df: pd.DataFrame, topk_values: Iterable[int]=(10, 25, 50, 100)) -> Dict[str, float]:
    out: Dict[str, float] = {}
    if pred_df.empty:
        return out
    y = pred_df['response_true'].astype(float).values
    yp = pred_df['response_pred'].astype(float).values
    out['response_rmse'] = float(np.sqrt(np.mean((y - yp) ** 2)))
    out['response_mae'] = float(np.mean(np.abs(y - yp)))
    out['response_spearman'] = float(spearmanr(y, yp).correlation) if len(y) > 1 and len(np.unique(y)) > 1 and (len(np.unique(yp)) > 1) else np.nan
    out['response_composite'] = float(out['response_spearman'] - out['response_rmse']) if np.isfinite(out['response_spearman']) else np.nan
    per = []
    for drug_id, sub in pred_df.groupby('drug_id'):
        yy = sub['response_true'].astype(float).values
        pp = sub['response_pred'].astype(float).values
        if len(yy) == 0:
            continue
        rmse = float(np.sqrt(np.mean((yy - pp) ** 2)))
        sp = float(spearmanr(yy, pp).correlation) if len(yy) > 1 and len(np.unique(yy)) > 1 and (len(np.unique(pp)) > 1) else np.nan
        per.append({'drug_id': drug_id, 'rmse': rmse, 'spearman': sp, 'n': len(yy)})
    per_df = pd.DataFrame(per)
    if not per_df.empty:
        out['response_rmse_drug_macro'] = float(per_df['rmse'].mean())
        out['response_rmse_drug_median'] = float(per_df['rmse'].median())
        out['response_spearman_drug_macro'] = float(per_df['spearman'].mean(skipna=True))
        out['response_spearman_drug_median'] = float(per_df['spearman'].median(skipna=True))
        out['response_n_drugs'] = int(per_df['drug_id'].nunique())
    out.update(topk_metrics(pred_df, topk_values))
    out.update(posthoc_selectivity_metrics_from_df(pred_df))
    out['response_discovery_composite'] = _finite_weighted_sum([(0.22, out.get('response_spearman_drug_macro', np.nan)), (0.2, out.get('top10_subtype_drug_ndcg_macro', np.nan)), (0.18, out.get('top10_drug_subtype_ndcg_macro', np.nan)), (0.15, out.get('top25_subtype_drug_ndcg_macro', np.nan)), (0.1, out.get('response_spearman', np.nan)), (0.1, out.get('posthoc_selectivity_spearman', np.nan)), (0.05, -out.get('response_rmse', np.nan))])
    return out

def ln_ic50_metrics_from_df(pred_df: pd.DataFrame, topk_values: Iterable[int]=(10, 25, 50, 100)) -> Dict[str, float]:
    out: Dict[str, float] = {}
    if pred_df.empty or 'ln_ic50_mask' not in pred_df.columns:
        return out
    df = pred_df[pred_df['ln_ic50_mask'].astype(float) > 0].copy()
    if df.empty:
        out['ln_ic50_rmse'] = np.nan
        out['ln_ic50_mae'] = np.nan
        out['ln_ic50_spearman'] = np.nan
        return out
    y = df['ln_ic50_true'].astype(float).values
    yp = df['ln_ic50_pred'].astype(float).values
    out['ln_ic50_rmse'] = float(np.sqrt(np.mean((y - yp) ** 2)))
    out['ln_ic50_mae'] = float(np.mean(np.abs(y - yp)))
    out['ln_ic50_spearman'] = float(spearmanr(y, yp).correlation) if len(y) > 1 and len(np.unique(y)) > 1 and (len(np.unique(yp)) > 1) else np.nan
    per = []
    for drug_id, sub in df.groupby('drug_id'):
        yy = sub['ln_ic50_true'].astype(float).values
        pp = sub['ln_ic50_pred'].astype(float).values
        if len(yy) == 0:
            continue
        rmse = float(np.sqrt(np.mean((yy - pp) ** 2)))
        sp = float(spearmanr(yy, pp).correlation) if len(yy) > 1 and len(np.unique(yy)) > 1 and (len(np.unique(pp)) > 1) else np.nan
        per.append({'drug_id': drug_id, 'rmse': rmse, 'spearman': sp, 'n': len(yy)})
    per_df = pd.DataFrame(per)
    if not per_df.empty:
        out['ln_ic50_rmse_drug_macro'] = float(per_df['rmse'].mean())
        out['ln_ic50_rmse_drug_median'] = float(per_df['rmse'].median())
        out['ln_ic50_spearman_drug_macro'] = float(per_df['spearman'].mean(skipna=True))
        out['ln_ic50_spearman_drug_median'] = float(per_df['spearman'].median(skipna=True))
        out['ln_ic50_n_drugs'] = int(per_df['drug_id'].nunique())
    true_order = np.argsort(df['ln_ic50_true'].astype(float).values)
    pred_order = np.argsort(df['ln_ic50_pred'].astype(float).values)
    for k in topk_values:
        p, r, n = _binary_topk_stats(true_order, pred_order, int(k))
        out[f'ln_ic50_top{k}_precision'] = p
        out[f'ln_ic50_top{k}_recall'] = r
        out[f'ln_ic50_top{k}_ndcg'] = n
    out['ln_ic50_discovery_composite'] = _finite_weighted_sum([(0.4, out.get('ln_ic50_spearman_drug_macro', np.nan)), (0.25, out.get('ln_ic50_spearman', np.nan)), (0.2, out.get('ln_ic50_top50_ndcg', np.nan)), (0.15, -out.get('ln_ic50_rmse', np.nan))])
    return out

def _binary_topk_stats(true_order: np.ndarray, pred_order: np.ndarray, k: int) -> Tuple[float, float, float]:
    n = len(true_order)
    if n == 0:
        return (np.nan, np.nan, np.nan)
    k = int(min(k, n))
    true_top = set(true_order[:k])
    pred_top = list(pred_order[:k])
    hits = np.array([1.0 if i in true_top else 0.0 for i in pred_top], dtype=float)
    precision = float(hits.mean()) if k > 0 else np.nan
    recall = float(hits.sum() / max(len(true_top), 1))
    discounts = 1.0 / np.log2(np.arange(2, k + 2))
    dcg = float((hits * discounts).sum())
    ideal_hits = np.ones(k, dtype=float)
    idcg = float((ideal_hits * discounts).sum())
    ndcg = float(dcg / idcg) if idcg > 0 else np.nan
    return (precision, recall, ndcg)

def _finite_weighted_sum(items: List[Tuple[float, float]]) -> float:
    vals = [(float(w), float(v)) for w, v in items if v is not None and np.isfinite(float(v)) and np.isfinite(float(w)) and (float(w) > 0)]
    if not vals:
        return np.nan
    total_w = sum((w for w, _ in vals))
    return float(sum((w * v for w, v in vals)) / max(total_w, 1e-12))

def topk_metrics(pred_df: pd.DataFrame, topk_values: Iterable[int]) -> Dict[str, float]:
    out: Dict[str, float] = {}
    if pred_df.empty:
        return out
    df = pred_df.reset_index(drop=True).copy()
    true_order = np.argsort(df['response_true'].astype(float).values)
    pred_order = np.argsort(df['response_pred'].astype(float).values)
    for k in topk_values:
        p, r, n = _binary_topk_stats(true_order, pred_order, int(k))
        out[f'top{k}_precision'] = p
        out[f'top{k}_recall'] = r
        out[f'top{k}_ndcg'] = n
    for k in topk_values:
        ps, ns = ([], [])
        for _, sub in df.groupby('drug_id'):
            if len(sub) < 2:
                continue
            idx = sub.reset_index(drop=True)
            kk = min(int(k), len(idx))
            true_o = np.argsort(idx['response_true'].astype(float).values)
            pred_o = np.argsort(idx['response_pred'].astype(float).values)
            p, _, nd = _binary_topk_stats(true_o, pred_o, kk)
            if np.isfinite(p):
                ps.append(p)
            if np.isfinite(nd):
                ns.append(nd)
        out[f'top{k}_drug_precision_macro'] = float(np.mean(ps)) if ps else np.nan
        out[f'top{k}_drug_ndcg_macro'] = float(np.mean(ns)) if ns else np.nan
    out.update(_grouped_topk_macro(df, 'tcga_desc', 'drug_id', 'response_true', 'response_pred', topk_values, 'subtype_drug'))
    out.update(_grouped_topk_macro(df, 'drug_id', 'tcga_desc', 'response_true', 'response_pred', topk_values, 'drug_subtype'))
    return out

def compute_losses(model: nn.Module, batch: Dict[str, Any], cfg: ConfigDruXRV5, aux_scale: float=1.0):
    _ensure_v5_config_defaults(cfg)
    out = model(batch)
    losses: Dict[str, torch.Tensor] = {}
    losses['response'] = F.smooth_l1_loss(out['response'], batch['response_target'], beta=cfg.response_huber_beta)
    losses['ln_ic50'] = masked_smooth_l1(out['ln_ic50'], batch['ln_ic50_target'], batch['ln_ic50_mask'], beta=cfg.ln_ic50_huber_beta)
    losses['tic'] = masked_smooth_l1(out['tic'], batch['tic_target'], batch['tic_mask'], beta=cfg.tic_huber_beta)
    losses['rank'] = informative_pairwise_rank_loss(out['rank'], batch['rank_target'], batch['rank_group'], batch.get('rank_mask'), min_delta=0.05, max_pairs=int(getattr(cfg, 'rank_max_pairs_per_group', 4096)))
    global_group = torch.zeros_like(batch['rank_group'])
    auc_score = -out['response']
    auc_target = -batch['response_target']
    ic50_score = -out['ln_ic50']
    ic50_target = -batch['ln_ic50_target']
    losses['response_pair_rank'] = informative_pairwise_rank_loss(auc_score, auc_target, batch['rank_group'], None, min_delta=float(getattr(cfg, 'rank_min_delta_auc', 0.05)), max_pairs=int(getattr(cfg, 'rank_max_pairs_per_group', 4096)))
    losses['response_global_rank'] = informative_pairwise_rank_loss(auc_score, auc_target, global_group, None, min_delta=float(getattr(cfg, 'rank_min_delta_auc', 0.05)), max_pairs=int(getattr(cfg, 'rank_max_pairs_per_group', 4096)))
    losses['response_subtype_listwise'] = group_listnet_rank_loss(auc_score, auc_target, batch['rank_group'], None, temperature=float(getattr(cfg, 'listwise_temperature', 0.15)), min_delta=float(getattr(cfg, 'rank_min_delta_auc', 0.05)))
    losses['response_drug_listwise'] = group_listnet_rank_loss(auc_score, auc_target, batch['drug_id'], None, temperature=float(getattr(cfg, 'listwise_temperature', 0.15)), min_delta=float(getattr(cfg, 'rank_min_delta_auc', 0.05)))
    losses['ln_ic50_pair_rank'] = informative_pairwise_rank_loss(ic50_score, ic50_target, batch['rank_group'], batch.get('ln_ic50_mask'), min_delta=float(getattr(cfg, 'rank_min_delta_ln_ic50', 0.25)), max_pairs=int(getattr(cfg, 'rank_max_pairs_per_group', 4096)))
    losses['ln_ic50_global_rank'] = informative_pairwise_rank_loss(ic50_score, ic50_target, global_group, batch.get('ln_ic50_mask'), min_delta=float(getattr(cfg, 'rank_min_delta_ln_ic50', 0.25)), max_pairs=int(getattr(cfg, 'rank_max_pairs_per_group', 4096)))
    losses['ln_ic50_subtype_listwise'] = group_listnet_rank_loss(ic50_score, ic50_target, batch['rank_group'], batch.get('ln_ic50_mask'), temperature=float(getattr(cfg, 'listwise_temperature', 0.15)), min_delta=float(getattr(cfg, 'rank_min_delta_ln_ic50', 0.25)))
    losses['ln_ic50_drug_listwise'] = group_listnet_rank_loss(ic50_score, ic50_target, batch['drug_id'], batch.get('ln_ic50_mask'), temperature=float(getattr(cfg, 'listwise_temperature', 0.15)), min_delta=float(getattr(cfg, 'rank_min_delta_ln_ic50', 0.25)))
    losses['moa'] = masked_bce_with_logits(out['moa'], batch['moa_target'], batch['moa_mask']) if 'moa' in out else torch.tensor(0.0, device=batch['response_target'].device)
    losses['selectivity'] = masked_bce_with_logits(out['selectivity'], batch['sel_target'], batch['sel_mask']) if 'selectivity' in out else torch.tensor(0.0, device=batch['response_target'].device)
    losses['pathway'] = masked_bce_with_logits(out['pathway'], batch['pathway_target'], batch['pathway_mask']) if 'pathway' in out else torch.tensor(0.0, device=batch['response_target'].device)
    losses['domain_adv'] = F.cross_entropy(out['domain'], batch['domain_target']) if 'domain' in out and cfg.lambda_domain_adv > 0 else torch.tensor(0.0, device=batch['response_target'].device)
    losses['total'] = cfg.lambda_response * losses['response'] + cfg.lambda_ln_ic50 * losses['ln_ic50'] + aux_scale * cfg.lambda_tic * losses['tic'] + aux_scale * cfg.lambda_moa * losses['moa'] + aux_scale * cfg.lambda_selectivity * losses['selectivity'] + aux_scale * cfg.lambda_rank * losses['rank'] + aux_scale * cfg.lambda_response_pair_rank * losses['response_pair_rank'] + aux_scale * cfg.lambda_response_global_rank * losses['response_global_rank'] + aux_scale * float(getattr(cfg, 'lambda_response_subtype_listwise', 0.0)) * losses['response_subtype_listwise'] + aux_scale * float(getattr(cfg, 'lambda_response_drug_listwise', 0.0)) * losses['response_drug_listwise'] + aux_scale * cfg.lambda_ln_ic50_pair_rank * losses['ln_ic50_pair_rank'] + aux_scale * cfg.lambda_ln_ic50_global_rank * losses['ln_ic50_global_rank'] + aux_scale * float(getattr(cfg, 'lambda_ln_ic50_subtype_listwise', 0.0)) * losses['ln_ic50_subtype_listwise'] + aux_scale * float(getattr(cfg, 'lambda_ln_ic50_drug_listwise', 0.0)) * losses['ln_ic50_drug_listwise'] + aux_scale * cfg.lambda_pathway * losses['pathway'] + aux_scale * cfg.lambda_domain_adv * losses['domain_adv']
    return (out, losses)

def train_one_epoch(model, loader, optimizer, scaler, device: str, cfg: ConfigDruXRV5, epoch: int):
    model.train()
    running: Dict[str, float] = {}
    aux_scale = 1.0 if cfg.aux_warmup_epochs <= 0 else min(1.0, float(epoch) / float(cfg.aux_warmup_epochs))
    for batch in loader:
        batch = move_batch_to_device(batch, device)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=cfg.use_amp and device.startswith('cuda')):
            _, losses = compute_losses(model, batch, cfg, aux_scale=aux_scale)
        scaler.scale(losses['total']).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()
        for k, v in losses.items():
            running[k] = running.get(k, 0.0) + float(v.detach().cpu())
    return {k: v / max(len(loader), 1) for k, v in running.items()}

def _fit_monotonic_linear_calibrator(pred_df: pd.DataFrame, true_col: str, pred_col: str) -> Dict[str, float]:
    if pred_df is None or pred_df.empty or true_col not in pred_df.columns or (pred_col not in pred_df.columns):
        return {'intercept': 0.0, 'slope': 1.0, 'identity': 1.0}
    df = pred_df[[true_col, pred_col]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(df) < 8 or df[pred_col].std(ddof=0) <= 1e-08:
        return {'intercept': 0.0, 'slope': 1.0, 'identity': 1.0}
    x = df[pred_col].astype(float).values
    y = df[true_col].astype(float).values
    slope, intercept = np.polyfit(x, y, 1)
    if not np.isfinite(slope) or not np.isfinite(intercept) or slope <= 0:
        return {'intercept': 0.0, 'slope': 1.0, 'identity': 1.0}
    return {'intercept': float(intercept), 'slope': float(slope), 'identity': 0.0}

def _apply_calibrator_to_column(df: pd.DataFrame, pred_col: str, cal: Dict[str, float], clip01: bool=False) -> pd.DataFrame:
    if df is None or df.empty or pred_col not in df.columns:
        return df
    out = df.copy()
    raw_col = pred_col + '_raw'
    if raw_col not in out.columns:
        out[raw_col] = out[pred_col]
    out[pred_col] = float(cal.get('intercept', 0.0)) + float(cal.get('slope', 1.0)) * pd.to_numeric(out[raw_col], errors='coerce')
    if clip01:
        out[pred_col] = out[pred_col].clip(lower=0.0, upper=1.0)
    return out

def _recompute_prediction_metrics(metrics: Dict[str, float], pred_df: pd.DataFrame, cfg: ConfigDruXRV5) -> Dict[str, float]:
    new_metrics = dict(metrics)
    new_metrics.update(response_metrics_from_df(pred_df, cfg.topk_values))
    new_metrics.update(ln_ic50_metrics_from_df(pred_df, cfg.topk_values))
    return new_metrics

def calibrate_val_test_predictions(val_pred: pd.DataFrame, test_pred: pd.DataFrame, cfg: ConfigDruXRV5):
    if not getattr(cfg, 'use_validation_calibration', False):
        return (val_pred, test_pred, {})
    calibrators: Dict[str, Dict[str, float]] = {}
    calibrators['response'] = _fit_monotonic_linear_calibrator(val_pred, 'response_true', 'response_pred')
    val_pred = _apply_calibrator_to_column(val_pred, 'response_pred', calibrators['response'], clip01=bool(getattr(cfg, 'clip_auc_predictions', True)))
    test_pred = _apply_calibrator_to_column(test_pred, 'response_pred', calibrators['response'], clip01=bool(getattr(cfg, 'clip_auc_predictions', True)))
    if 'ln_ic50_mask' in val_pred.columns:
        val_ic = val_pred[val_pred['ln_ic50_mask'].astype(float) > 0].copy()
    else:
        val_ic = val_pred
    calibrators['ln_ic50'] = _fit_monotonic_linear_calibrator(val_ic, 'ln_ic50_true', 'ln_ic50_pred')
    val_pred = _apply_calibrator_to_column(val_pred, 'ln_ic50_pred', calibrators['ln_ic50'], clip01=False)
    test_pred = _apply_calibrator_to_column(test_pred, 'ln_ic50_pred', calibrators['ln_ic50'], clip01=False)
    return (val_pred, test_pred, calibrators)

def fit_model(model, train_loader, val_loader, test_loader, cfg: ConfigDruXRV5, output_dir: str, prefix: str):
    device = cfg.device
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode=cfg.monitor_mode, factor=cfg.scheduler_factor, patience=cfg.scheduler_patience)
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_amp and device.startswith('cuda'))
    best_score = float('inf') if cfg.monitor_mode == 'min' else float('-inf')
    best_state, best_epoch, best_val_metrics = (None, 0, None)
    history, patience_count = ([], 0)
    for epoch in range(1, cfg.epochs + 1):
        train_metrics = train_one_epoch(model, train_loader, optimizer, scaler, device, cfg, epoch)
        val_metrics, _ = eval_one_epoch(model, val_loader, device, cfg)
        monitor_value = val_metrics.get(cfg.monitor_metric, val_metrics.get('total', np.nan))
        if not np.isfinite(monitor_value):
            monitor_value = val_metrics.get('total', np.nan)
        scheduler.step(monitor_value)
        row = {'epoch': epoch}
        row.update({f'train_{k}': v for k, v in train_metrics.items()})
        row.update({f'val_{k}': v for k, v in val_metrics.items()})
        history.append(row)
        print(f"[{prefix}][{epoch:02d}] train_total={train_metrics.get('total', np.nan):.4f} val_total={val_metrics.get('total', np.nan):.4f} val_resp_rmse={val_metrics.get('response_rmse', np.nan):.4f} val_resp_rmse_drug_macro={val_metrics.get('response_rmse_drug_macro', np.nan):.4f} val_resp_spearman={val_metrics.get('response_spearman', np.nan):.4f} val_ln_ic50_rmse={val_metrics.get('ln_ic50_rmse', np.nan):.4f} val_ln_ic50_spearman={val_metrics.get('ln_ic50_spearman', np.nan):.4f} monitor={monitor_value:.4f}")
        improved = monitor_value < best_score if cfg.monitor_mode == 'min' else monitor_value > best_score
        if improved:
            best_score = float(monitor_value)
            best_epoch = epoch
            best_val_metrics = dict(val_metrics)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_count = 0
            if cfg.save_checkpoints:
                torch.save(best_state, os.path.join(output_dir, f'{prefix}_best.pt'))
        else:
            patience_count += 1
            if patience_count >= cfg.early_stop_patience:
                print(f'[{prefix}] Early stopping at epoch {epoch}.')
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    reloaded_val_metrics, val_pred = eval_one_epoch(model, val_loader, device, cfg)
    test_metrics, test_pred = eval_one_epoch(model, test_loader, device, cfg)
    if isinstance(val_pred, pd.DataFrame) and (not val_pred.empty):
        val_pred['split'] = 'val'
    if isinstance(test_pred, pd.DataFrame) and (not test_pred.empty):
        test_pred['split'] = 'test'
    val_pred, test_pred, calibrators = calibrate_val_test_predictions(val_pred, test_pred, cfg)
    reloaded_val_metrics = _recompute_prediction_metrics(reloaded_val_metrics, val_pred, cfg)
    test_metrics = _recompute_prediction_metrics(test_metrics, test_pred, cfg)
    val_metrics = reloaded_val_metrics
    history_df = pd.DataFrame(history)
    if cfg.save_history:
        history_df.to_csv(os.path.join(output_dir, f'{prefix}_history.csv'), index=False)
    if cfg.save_predictions:
        val_pred.to_csv(os.path.join(output_dir, f'{prefix}_val_predictions.csv'), index=False)
        test_pred.to_csv(os.path.join(output_dir, f'{prefix}_test_predictions.csv'), index=False)
        per_drug_response_table(test_pred).to_csv(os.path.join(output_dir, f'{prefix}_test_per_drug_response.csv'), index=False)
    summary = {'best_epoch': best_epoch, 'best_monitor_metric': cfg.monitor_metric, 'best_monitor_score': best_score, 'best_val_total': float(val_metrics.get('total', np.nan)), 'val_metrics': val_metrics, 'reloaded_val_metrics': reloaded_val_metrics, 'test_metrics': test_metrics, 'calibrators': calibrators if 'calibrators' in locals() else {}}
    with open(os.path.join(output_dir, f'{prefix}_summary.json'), 'w') as f:
        json.dump(summary, f, indent=2, default=str)
    return {'model': model, 'history': history_df, 'val_predictions': val_pred, 'test_predictions': test_pred, 'summary': summary}

def per_drug_response_table(pred_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for drug_id, sub in pred_df.groupby('drug_id'):
        y = sub['response_true'].astype(float).values
        yp = sub['response_pred'].astype(float).values
        row = {'drug_id': drug_id, 'n_rows': len(sub), 'auc_rmse': float(np.sqrt(np.mean((y - yp) ** 2))) if len(y) else np.nan, 'auc_mae': float(np.mean(np.abs(y - yp))) if len(y) else np.nan, 'auc_spearman': float(spearmanr(y, yp).correlation) if len(y) > 1 and len(np.unique(y)) > 1 and (len(np.unique(yp)) > 1) else np.nan, 'true_auc_mean': float(np.mean(y)) if len(y) else np.nan, 'pred_auc_mean': float(np.mean(yp)) if len(yp) else np.nan}
        if {'ln_ic50_true', 'ln_ic50_pred', 'ln_ic50_mask'}.issubset(sub.columns):
            sub_ic50 = sub[sub['ln_ic50_mask'].astype(float) > 0]
            yy = sub_ic50['ln_ic50_true'].astype(float).values
            pp = sub_ic50['ln_ic50_pred'].astype(float).values
            row.update({'ln_ic50_n_rows': int(len(sub_ic50)), 'ln_ic50_rmse': float(np.sqrt(np.mean((yy - pp) ** 2))) if len(yy) else np.nan, 'ln_ic50_mae': float(np.mean(np.abs(yy - pp))) if len(yy) else np.nan, 'ln_ic50_spearman': float(spearmanr(yy, pp).correlation) if len(yy) > 1 and len(np.unique(yy)) > 1 and (len(np.unique(pp)) > 1) else np.nan, 'true_ln_ic50_mean': float(np.mean(yy)) if len(yy) else np.nan, 'pred_ln_ic50_mean': float(np.mean(pp)) if len(pp) else np.nan})
        rows.append(row)
    return pd.DataFrame(rows)

def make_loaders_and_model_for_seed(graph_pairs_df: pd.DataFrame, drug_graph_df: pd.DataFrame, expr_graph_df: pd.DataFrame, expr_cols: List[str], domain_info: Dict[str, Any], cfg: ConfigDruXRV5, output_dir: str):
    tr_df, va_df, te_df, tr_ids, va_ids, te_ids = split_by_drug(graph_pairs_df, cfg.train_frac, cfg.val_frac, cfg.test_frac, cfg.split_seed)
    print(f'[strict_v5] Split drugs: train={len(tr_ids)} val={len(va_ids)} test={len(te_ids)}')
    expr_scaler, expr_pca, expr_embed_df, expr_raw_df, expr_gene_set = fit_transform_expression_train_only(expr_graph_df, expr_cols, tr_df, cfg)
    drug_feat_df, drug_feature_cols = fit_transform_drug_features_train_only(drug_graph_df, tr_ids, cfg)
    tr_df, va_df, te_df, moa_targets_df, moa_vocab, pathway_targets_df, pathway_vocab, sel_targets_df, subtype_vocab, sel_train_valid_ids = prepare_split_auxiliary_targets(tr_df, va_df, te_df, drug_graph_df, tr_ids, cfg)
    drug_graph_dict, smiles_source_col = build_graph_drug_dict(drug_graph_df)
    valid_ids = sorted(set(drug_graph_dict.keys()))
    tr_df = tr_df[tr_df['DRUG_ID'].astype(int).isin(valid_ids)].reset_index(drop=True)
    va_df = va_df[va_df['DRUG_ID'].astype(int).isin(valid_ids)].reset_index(drop=True)
    te_df = te_df[te_df['DRUG_ID'].astype(int).isin(valid_ids)].reset_index(drop=True)
    drug_graph_df = drug_graph_df[drug_graph_df['DRUG_ID'].astype(int).isin(valid_ids)].reset_index(drop=True)
    domain_vocab = sorted(graph_pairs_df['DOMAIN_COMBINED'].dropna().astype(str).unique().tolist())
    domain_to_id = {d: i for i, d in enumerate(domain_vocab)}
    rank_group_vocab = sorted(tr_df['rank_group'].fillna('__UNK__').astype(str).unique().tolist())
    rank_group_to_id = {g: i for i, g in enumerate(rank_group_vocab)}
    enable_domain_adv = bool(domain_info.get('true_multi_domain', False)) and cfg.lambda_domain_adv > 0
    ds_kwargs = dict(drug_df_full=drug_graph_df, drug_feat_df=drug_feat_df, drug_feature_cols=drug_feature_cols, expr_embed_df=expr_embed_df, expr_raw_df=expr_raw_df, expr_columns_set=expr_gene_set, drug_graph_dict=drug_graph_dict, moa_targets_df=moa_targets_df, selectivity_targets_df=sel_targets_df, pathway_targets_df=pathway_targets_df, domain_to_id=domain_to_id, rank_group_to_id=rank_group_to_id, selectivity_train_valid_ids=sel_train_valid_ids, cfg=cfg)
    train_ds = GraphPairDatasetV2Strict(tr_df, **ds_kwargs)
    val_ds = GraphPairDatasetV2Strict(va_df, **ds_kwargs)
    test_ds = GraphPairDatasetV2Strict(te_df, **ds_kwargs)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, collate_fn=collate_graph_batch_v2_strict)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, collate_fn=collate_graph_batch_v2_strict)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, collate_fn=collate_graph_batch_v2_strict)
    model = DruXRGraphV2Strict(drug_tab_dim=len(drug_feature_cols), cell_dim=expr_embed_df.shape[1], tic_dim=9, n_moa=len(moa_vocab), n_sel=len(subtype_vocab), n_pathway=len(pathway_vocab), n_domains=len(domain_vocab), enable_domain_adv=enable_domain_adv, cfg=cfg)
    meta = {'train_drug_ids': tr_ids, 'val_drug_ids': va_ids, 'test_drug_ids': te_ids, 'smiles_source_col': smiles_source_col, 'moa_vocab': moa_vocab, 'pathway_vocab': pathway_vocab, 'subtype_vocab': subtype_vocab, 'domain_vocab': domain_vocab, 'rank_group_vocab': rank_group_vocab, 'drug_feature_cols': drug_feature_cols, 'strict_notes': {'pathway_leakage_removed': bool(cfg.remove_pathway_feature_leakage), 'expression_preprocessing': 'StandardScaler + PCA fit on train cells only', 'drug_features': 'median/imputation/scale fit on train drugs only', 'rank_selectivity_auxiliary': 'train pairs only; val/test rank/selectivity masks are zero', 'moa_pathway_vocab': 'built from train drugs only', 'dual_response_heads': 'AUC response head plus masked LN_IC50 response head', 'fixed_split_seed': int(cfg.split_seed), 'direct_pairwise_ranking_losses': 'response and LN_IC50 pairwise/global ranking losses on train labels only', 'validation_calibration': bool(cfg.use_validation_calibration)}}
    return (train_loader, val_loader, test_loader, model, meta)

def _run_one_v5_split_seed(base_cfg: ConfigDruXRV5, all_df: Optional[pd.DataFrame], seed: int, root_output_dir: str, split_seed: Optional[int]=None):
    cfg = ConfigDruXRV5(**asdict(base_cfg))
    for k, v in vars(base_cfg).items():
        if not hasattr(cfg, k):
            setattr(cfg, k, v)
    _ensure_v5_config_defaults(cfg)
    cfg.seed = int(seed)
    if split_seed is not None:
        cfg.split_seed = int(split_seed)
    set_seed(cfg.seed)
    seed_dir = os.path.join(root_output_dir, f'split{cfg.split_seed}_seed{seed}')
    ensure_dir(seed_dir)
    graph_pairs_df, drug_graph_df, expr_graph_df, expr_cols = load_input_data(cfg, all_df=all_df)
    print('graph_pairs_df:', graph_pairs_df.shape)
    print('drug_graph_df:', drug_graph_df.shape)
    print('expr_graph_df:', expr_graph_df.shape)
    print('# expression columns:', len(expr_cols))
    domain_info = detect_domain_mode(graph_pairs_df, cfg.allow_medium_only_domain_adv)
    print('Domain summary:')
    print(domain_info)
    if domain_info['medium_only_variation'] and (not domain_info['true_multi_domain']):
        print('[INFO] Medium-only variation detected. Domain adversarial consistency will remain OFF by default.')
    train_loader, val_loader, test_loader, model, meta = make_loaders_and_model_for_seed(graph_pairs_df, drug_graph_df, expr_graph_df, expr_cols, domain_info, cfg, seed_dir)
    with open(os.path.join(seed_dir, 'strict_v5_graph_only_dual_seed_metadata.json'), 'w') as f:
        json.dump({'config': dict(vars(cfg)), 'domain_info': domain_info, **meta}, f, indent=2, default=str)
    result = fit_model(model, train_loader, val_loader, test_loader, cfg, seed_dir, prefix='druxr_graph_v5_graph_only_strict_dual')
    for pred_key in ['val_predictions', 'test_predictions']:
        pred_df = result.get(pred_key)
        if isinstance(pred_df, pd.DataFrame) and (not pred_df.empty):
            pred_df['split_seed'] = int(cfg.split_seed)
            pred_df['model_seed'] = int(cfg.seed)
            pred_df['seed'] = int(cfg.seed)
            pred_df['run_key'] = f'split{cfg.split_seed}_seed{cfg.seed}'
            result[pred_key] = pred_df
            split_label = 'val' if pred_key == 'val_predictions' else 'test'
            pred_df.to_csv(os.path.join(seed_dir, f'druxr_graph_v5_graph_only_strict_dual_{split_label}_predictions.csv'), index=False)
    result['metadata'] = meta
    result['output_dir'] = seed_dir
    result['split_seed'] = int(cfg.split_seed)
    result['model_seed'] = int(cfg.seed)
    result['run_key'] = f'split{cfg.split_seed}_seed{cfg.seed}'
    return result

def summarize_seed_metrics(metrics_df: pd.DataFrame):
    if metrics_df.empty:
        return (pd.DataFrame(), {})
    row = {'branch': 'graph_v5_graph_only_strict_dual', 'n_seeds': int(metrics_df['seed'].nunique())}
    if 'split_seed' in metrics_df.columns:
        row['n_split_seeds'] = int(metrics_df['split_seed'].nunique())
        row['n_runs'] = int(len(metrics_df))
    for col in metrics_df.columns:
        if col in {'seed', 'branch'}:
            continue
        if pd.api.types.is_numeric_dtype(metrics_df[col]):
            row[f'{col}_mean'] = float(metrics_df[col].mean())
            row[f'{col}_std'] = float(metrics_df[col].std(ddof=1)) if metrics_df[col].dropna().shape[0] > 1 else 0.0
    summary_df = pd.DataFrame([row])
    return (summary_df, row)

def _run_single_specialist_base(cfg: Optional[ConfigDruXRV5]=None, all_df: Optional[pd.DataFrame]=None):
    if cfg is None:
        cfg = CFG_DRUXR_V5
    root_output_dir = os.path.join(cfg.output_dir, cfg.run_name)
    ensure_dir(root_output_dir)
    seed_results: Dict[str, Dict[str, Any]] = {}
    metric_rows: List[Dict[str, Any]] = []
    active_split_seed_values = tuple(getattr(cfg, 'split_seeds', ()) or (getattr(cfg, 'split_seed', 1),))
    model_seeds = tuple(getattr(cfg, 'seeds', ()) or (getattr(cfg, 'seed', 1),))
    for split_seed in active_split_seed_values:
        for seed in model_seeds:
            run_key = f'split{int(split_seed)}_seed{int(seed)}'
            print(f'\n================ DruXR v5 GRAPH-ONLY SPLIT {split_seed} | MODEL SEED {seed} ================')
            result = _run_one_v5_split_seed(cfg, all_df=all_df, seed=int(seed), root_output_dir=root_output_dir, split_seed=int(split_seed))
            seed_results[run_key] = result
            row = {'run_key': run_key, 'split_seed': int(split_seed), 'seed': int(seed), 'model_seed': int(seed), 'branch': 'graph_v5_clean_strict_dual'}
            test_metrics = result['summary'].get('test_metrics', {})
            val_metrics = result['summary'].get('val_metrics', {})
            for k, v in test_metrics.items():
                if isinstance(v, (int, float, np.integer, np.floating)):
                    row[f'test_{k}'] = float(v)
            for k, v in val_metrics.items():
                if isinstance(v, (int, float, np.integer, np.floating)):
                    row[f'val_{k}'] = float(v)
            metric_rows.append(row)
    metrics_df = pd.DataFrame(metric_rows)
    summary_df, summary = summarize_seed_metrics(metrics_df)
    metrics_df.to_csv(os.path.join(root_output_dir, 'druxr_v5_single_specialist_seed_metrics.csv'), index=False)
    summary_df.to_csv(os.path.join(root_output_dir, 'druxr_v5_single_specialist_summary.csv'), index=False)
    with open(os.path.join(root_output_dir, 'druxr_v5_single_specialist_config.json'), 'w') as f:
        json.dump(asdict(cfg), f, indent=2, default=str)
    print('\nDruXR v5 single-specialist run complete.')
    print('Artifacts saved under:', root_output_dir)
    print('Summary mean ± std saved in druxr_v5_single_specialist_summary.csv')
    print(summary)
    return {'seed_results': seed_results, 'metrics': metrics_df, 'summary': summary_df, 'summary_dict': summary, 'output_dir': root_output_dir, 'config': cfg}

def _bh_fdr(p_values: Iterable[float]) -> np.ndarray:
    p = np.array([np.nan if v is None else float(v) for v in p_values], dtype=float)
    q = np.full_like(p, np.nan, dtype=float)
    valid = np.isfinite(p)
    if valid.sum() == 0:
        return q
    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]
    m = float(len(ranked))
    raw = ranked * m / (np.arange(len(ranked)) + 1.0)
    adj = np.minimum.accumulate(raw[::-1])[::-1]
    adj = np.clip(adj, 0.0, 1.0)
    out = np.empty_like(pv)
    out[order] = adj
    q[valid] = out
    return q

def _safe_numeric_zscore(s: pd.Series, invert: bool=False) -> pd.Series:
    x = pd.to_numeric(s, errors='coerce')
    std = x.std(ddof=0)
    if pd.isna(std) or std == 0:
        z = pd.Series(np.zeros(len(x)), index=x.index, dtype=float)
    else:
        z = (x - x.mean()) / std
    return -z if invert else z

def _score_to_percentile_good(series: pd.Series, lower_is_better: bool=True) -> pd.Series:
    x = pd.to_numeric(series, errors='coerce')
    if x.notna().sum() <= 1:
        return pd.Series(np.ones(len(x)) * 0.5, index=x.index)
    ranks = x.rank(method='average', pct=True, ascending=lower_is_better)
    if lower_is_better:
        return 1.0 - ranks + 1.0 / max(x.notna().sum(), 1)
    return ranks
STRICT_DISCOVERY_AUC_MAX = 0.7
STRICT_DISCOVERY_AUC_STRONG = 0.65
STRICT_DISCOVERY_RANK_MAX = 5
STRICT_DISCOVERY_RANK_STRONG = 3
STRICT_DISCOVERY_DELTA_MIN = 0.03
STRICT_DISCOVERY_DELTA_STRONG = 0.05
STRICT_DISCOVERY_FDR = 0.05
STRICT_DISCOVERY_TIC_MIN = 0.7
STRICT_DISCOVERY_BIOLOGY_MIN = 0.5

def _clean_text_value(v: Any) -> str:
    if v is None or pd.isna(v):
        return ''
    s = str(v).strip()
    if s.lower() in {'', 'nan', 'none', '<na>', 'not available', 'unknown'}:
        return ''
    return s

def _valid_subtype(s: Any) -> bool:
    t = _clean_text_value(s).upper()
    return t not in {'', 'UNKNOWN', 'NAN', 'UNCLASSIFIED', 'NONE'}

def _valid_moa_label(s: Any) -> bool:
    t = _clean_text_value(s).lower()
    return t not in {'', 'unclassified', 'other', 'unknown', 'not available'}

def _valid_pathway_label(s: Any) -> bool:
    t = _clean_text_value(s).lower()
    return t not in {'', 'unclassified', 'other', 'unknown', 'not available'}

def _valid_target_label(s: Any) -> bool:
    t = _clean_text_value(s).lower()
    return t not in {'', 'unclassified', 'other', 'unknown', 'not available'}

def _clip01(x: Any) -> float:
    try:
        v = float(x)
    except Exception:
        return 0.0
    if not np.isfinite(v):
        return 0.0
    return float(np.clip(v, 0.0, 1.0))

def _rank_score_from_rank(rank: Any, max_rank: int=STRICT_DISCOVERY_RANK_MAX) -> float:
    try:
        r = float(rank)
    except Exception:
        return 0.0
    if not np.isfinite(r) or r <= 0:
        return 0.0
    return float(np.clip((float(max_rank) + 1.0 - r) / float(max_rank), 0.0, 1.0))

def _first_nonempty_for_report(series: pd.Series):
    for v in series:
        if pd.notna(v) and str(v).strip() not in {'', 'nan', 'None', '<NA>'}:
            return str(v)
    return ''

def build_discovery_drug_annotations(all_df: pd.DataFrame) -> pd.DataFrame:
    if all_df is None or not isinstance(all_df, pd.DataFrame) or 'DRUG_ID' not in all_df.columns:
        return pd.DataFrame(columns=['drug_id', 'drug_name', 'moa_label', 'pathway_label', 'target_label'])
    df = all_df.copy()
    name_col = pick_first_available(df, ['DRUG_NAME', 'COMPOUND_NAME', 'chembl_pref_name', 'pubchem_match_name'], default=None)
    moa_col = pick_first_available(df, ['MOA_LABEL_COLLAPSED_TRAIN', 'MOA_LABEL_COLLAPSED', 'MOA_LABEL_FINAL', 'TARGET'], default=None)
    path_col = pick_first_available(df, ['PATHWAY_NAME_FINAL', 'PATHWAY_NAME'], default=None)
    target_col = pick_first_available(df, ['TARGET_GENES_FINAL', 'target_genes_in_expr_str', 'TARGET'], default=None)
    keep = ['DRUG_ID'] + [c for c in [name_col, moa_col, path_col, target_col] if c is not None]
    sub = df[keep].copy().drop_duplicates()
    agg = {}
    for c in sub.columns:
        if c != 'DRUG_ID':
            agg[c] = _first_nonempty_for_report
    ann = sub.groupby('DRUG_ID', as_index=False).agg(agg)
    ann = ann.rename(columns={'DRUG_ID': 'drug_id'})
    if name_col and name_col in ann.columns:
        ann = ann.rename(columns={name_col: 'drug_name'})
    else:
        ann['drug_name'] = ''
    if moa_col and moa_col in ann.columns:
        ann = ann.rename(columns={moa_col: 'moa_label'})
    else:
        ann['moa_label'] = ''
    if path_col and path_col in ann.columns:
        ann = ann.rename(columns={path_col: 'pathway_label'})
    else:
        ann['pathway_label'] = ''
    if target_col and target_col in ann.columns:
        ann = ann.rename(columns={target_col: 'target_label'})
    else:
        ann['target_label'] = ''
    return ann[['drug_id', 'drug_name', 'moa_label', 'pathway_label', 'target_label']]

def _combine_test_predictions_from_result(result: Dict[str, Any]) -> pd.DataFrame:
    frames = []
    for run_key, seed_result in result.get('seed_results', {}).items():
        pred = seed_result.get('test_predictions')
        if isinstance(pred, pd.DataFrame) and (not pred.empty):
            tmp = pred.copy()
            if 'seed' not in tmp.columns:
                tmp['seed'] = int(seed_result.get('model_seed', seed_result.get('seed', 1)))
            if 'model_seed' not in tmp.columns:
                tmp['model_seed'] = tmp['seed']
            if 'split_seed' not in tmp.columns:
                tmp['split_seed'] = int(seed_result.get('split_seed', 1))
            if 'run_key' not in tmp.columns:
                tmp['run_key'] = str(run_key)
            frames.append(tmp)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def build_subtype_candidate_table(pred_df: pd.DataFrame, all_df: Optional[pd.DataFrame]=None) -> pd.DataFrame:
    if pred_df is None or pred_df.empty:
        return pd.DataFrame()
    df = pred_df.copy()
    if 'tcga_desc' not in df.columns:
        df['tcga_desc'] = 'UNKNOWN'
    df['tcga_desc'] = df['tcga_desc'].fillna('UNKNOWN').astype(str)
    df['drug_id'] = df['drug_id'].astype(int)
    for c in ['response_pred', 'response_true', 'rank_pred', 'tic_pred', 'ln_ic50_pred', 'moa_pred_max', 'pathway_pred_max', 'selectivity_pred_max']:
        if c not in df.columns:
            df[c] = np.nan
        df[c] = pd.to_numeric(df[c], errors='coerce')
    if 'ln_ic50_mask' not in df.columns:
        df['ln_ic50_mask'] = 0.0
    df['ln_ic50_mask'] = pd.to_numeric(df['ln_ic50_mask'], errors='coerce').fillna(0.0)
    if 'split_seed' in df.columns and 'seed' in df.columns:
        group_cols = ['split_seed', 'seed', 'drug_id', 'tcga_desc']
    elif 'seed' in df.columns:
        group_cols = ['seed', 'drug_id', 'tcga_desc']
    else:
        group_cols = ['drug_id', 'tcga_desc']
    cand = df.groupby(group_cols, as_index=False).agg(n_cell_lines=('cell_line', 'nunique'), pred_auc_mean=('response_pred', 'mean'), pred_auc_median=('response_pred', 'median'), true_auc_mean=('response_true', 'mean'), rank_pred_mean=('rank_pred', 'mean'), tic_pred_mean=('tic_pred', 'mean'), pred_ln_ic50_mean=('ln_ic50_pred', 'mean'), ln_ic50_available=('ln_ic50_mask', 'max'), moa_pred_max_mean=('moa_pred_max', 'mean'), pathway_pred_max_mean=('pathway_pred_max', 'mean'), selectivity_pred_max_mean=('selectivity_pred_max', 'mean'))
    cand['valid_subtype'] = cand['tcga_desc'].map(_valid_subtype)
    run_cols = [c for c in ['split_seed', 'seed'] if c in df.columns]
    cand_run_cols = [c for c in run_cols if c in cand.columns]
    group_cols = cand_run_cols + ['drug_id', 'tcga_desc']
    sort_cols = cand_run_cols + ['tcga_desc'] if cand_run_cols else ['tcga_desc']
    cand['subtype_auc_rank'] = cand.groupby(sort_cols, dropna=False)['pred_auc_mean'].rank(method='dense', ascending=True)
    cand['subtype_auc_percentile_good'] = cand.groupby(sort_cols, dropna=False)['pred_auc_mean'].transform(lambda s: _score_to_percentile_good(s, lower_is_better=True))
    cand['subtype_rank_score_percentile'] = cand.groupby(sort_cols, dropna=False)['rank_pred_mean'].transform(lambda s: _score_to_percentile_good(s, lower_is_better=False))
    rows = []
    raw_group = run_cols + ['drug_id']
    for key, sub in df.groupby(raw_group, dropna=False):
        key_tuple = key if isinstance(key, tuple) else (key,)
        key_dict = dict(zip(raw_group, key_tuple))
        for subtype, sub_st in sub.groupby('tcga_desc', dropna=False):
            other = sub[sub['tcga_desc'] != subtype]
            if other.empty or len(sub_st) < 2 or len(other) < 2:
                delta = np.nan
                pval = np.nan
                other_mean = np.nan
            else:
                delta = float(other['response_pred'].mean() - sub_st['response_pred'].mean())
                other_mean = float(other['response_pred'].mean())
                try:
                    pval = float(mannwhitneyu(sub_st['response_pred'], other['response_pred'], alternative='less').pvalue)
                except Exception:
                    pval = np.nan
            row = {'drug_id': int(key_dict['drug_id']), 'tcga_desc': subtype, 'selectivity_delta_auc': delta, 'other_subtypes_pred_auc_mean': other_mean, 'selectivity_pvalue': pval}
            for c in cand_run_cols:
                if c in key_dict:
                    row[c] = int(key_dict[c])
            rows.append(row)
    stats = pd.DataFrame(rows)
    if not stats.empty:
        stats['selectivity_fdr'] = _bh_fdr(stats['selectivity_pvalue'].values)
        merge_cols = [c for c in group_cols if c in cand.columns and c in stats.columns]
        cand = cand.merge(stats, on=merge_cols, how='left')
    else:
        cand['selectivity_delta_auc'] = np.nan
        cand['other_subtypes_pred_auc_mean'] = np.nan
        cand['selectivity_pvalue'] = np.nan
        cand['selectivity_fdr'] = np.nan
    ann = build_discovery_drug_annotations(all_df) if all_df is not None else pd.DataFrame()
    if not ann.empty:
        cand = cand.merge(ann, on='drug_id', how='left')
    for c in ['drug_name', 'moa_label', 'pathway_label', 'target_label']:
        if c not in cand.columns:
            cand[c] = ''
        cand[c] = cand[c].fillna('').astype(str)
    cand['has_moa_annotation'] = cand['moa_label'].map(_valid_moa_label)
    cand['has_pathway_annotation'] = cand['pathway_label'].map(_valid_pathway_label)
    cand['has_target_annotation'] = cand['target_label'].map(_valid_target_label)
    cand['fdr_pass'] = cand['selectivity_fdr'].le(STRICT_DISCOVERY_FDR)
    cand['fdr_trend'] = cand['selectivity_fdr'].le(0.1)
    cand['tic_support_score'] = cand.groupby(sort_cols)['tic_pred_mean'].transform(lambda s: _score_to_percentile_good(s, lower_is_better=False))
    med_delta = cand['selectivity_delta_auc'].median(skipna=True)
    if not np.isfinite(med_delta):
        med_delta = 0.0
    cand['selectivity_score'] = _score_to_percentile_good(cand['selectivity_delta_auc'].fillna(med_delta), lower_is_better=False)
    cand['moa_support_score'] = cand['moa_pred_max_mean'].map(_clip01)
    cand['pathway_support_score'] = cand['pathway_pred_max_mean'].map(_clip01)
    cand['biology_support_score'] = np.maximum(cand['moa_support_score'], cand['pathway_support_score'])
    cand.loc[~(cand['has_moa_annotation'] | cand['has_pathway_annotation']), 'biology_support_score'] = 0.0
    cand['low_auc_pass'] = cand['pred_auc_mean'].le(STRICT_DISCOVERY_AUC_MAX)
    cand['strong_auc_pass'] = cand['pred_auc_mean'].le(STRICT_DISCOVERY_AUC_STRONG)
    cand['top_rank_pass'] = cand['subtype_auc_rank'].le(STRICT_DISCOVERY_RANK_MAX)
    cand['strong_rank_pass'] = cand['subtype_auc_rank'].le(STRICT_DISCOVERY_RANK_STRONG)
    cand['selectivity_delta_pass'] = cand['selectivity_delta_auc'].ge(STRICT_DISCOVERY_DELTA_MIN)
    cand['selectivity_delta_strong'] = cand['selectivity_delta_auc'].ge(STRICT_DISCOVERY_DELTA_STRONG)
    cand['tic_support_pass'] = cand['tic_support_score'].ge(STRICT_DISCOVERY_TIC_MIN) & cand['has_target_annotation']
    cand['biology_support_pass'] = cand['biology_support_score'].ge(STRICT_DISCOVERY_BIOLOGY_MIN) & (cand['has_moa_annotation'] | cand['has_pathway_annotation'])
    cand['response_score'] = cand['subtype_auc_percentile_good'].fillna(0.0).map(_clip01)
    cand['rank_score'] = cand['subtype_auc_rank'].map(lambda r: _rank_score_from_rank(r, STRICT_DISCOVERY_RANK_MAX))
    cand['delta_score'] = (pd.to_numeric(cand['selectivity_delta_auc'], errors='coerce').fillna(0.0) / STRICT_DISCOVERY_DELTA_STRONG).clip(lower=0.0, upper=1.0)
    cand['fdr_score'] = cand['fdr_pass'].astype(float)
    cand['mechanism_score'] = np.maximum(cand['biology_support_score'].fillna(0.0), cand['tic_support_score'].fillna(0.0) * cand['has_target_annotation'].astype(float))
    cand['discovery_score'] = 0.45 * cand['response_score'] + 0.2 * cand['rank_score'] + 0.15 * cand['delta_score'] + 0.1 * cand['fdr_score'] + 0.05 * cand['biology_support_score'].fillna(0.0) + 0.05 * cand['tic_support_score'].fillna(0.0) * cand['has_target_annotation'].astype(float)
    cand['strict_candidate_pass'] = cand['valid_subtype'] & cand['low_auc_pass'] & cand['top_rank_pass'] & cand['selectivity_delta_pass'] & cand['fdr_pass'] & (cand['biology_support_pass'] | cand['tic_support_pass'] | cand['has_pathway_annotation'] | cand['has_target_annotation'])
    cand['strong_candidate_pass'] = cand['valid_subtype'] & cand['strong_auc_pass'] & cand['strong_rank_pass'] & cand['selectivity_delta_strong'] & cand['fdr_pass'] & (cand['biology_support_pass'] | cand['tic_support_pass'] | cand['has_pathway_annotation'] | cand['has_target_annotation'])
    return cand.sort_values('discovery_score', ascending=False).reset_index(drop=True)

def _external_replication_status(row: pd.Series, external_df: Optional[pd.DataFrame]=None) -> str:
    if external_df is None or not isinstance(external_df, pd.DataFrame) or external_df.empty:
        return 'External/experimental replication: not assessed; no external validation table supplied.'
    ext = external_df.copy()
    if 'drug_id' not in ext.columns and 'DRUG_ID' in ext.columns:
        ext = ext.rename(columns={'DRUG_ID': 'drug_id'})
    if 'tcga_desc' not in ext.columns:
        for alt in ['TCGA_DESC', 'subtype', 'cancer_subtype']:
            if alt in ext.columns:
                ext = ext.rename(columns={alt: 'tcga_desc'})
                break
    if 'drug_id' not in ext.columns or 'tcga_desc' not in ext.columns:
        return 'External/experimental replication: not assessed; external table lacks drug_id/tcga_desc columns.'
    m = ext[(ext['drug_id'].astype(str) == str(row.get('drug_id'))) & (ext['tcga_desc'].astype(str) == str(row.get('tcga_desc')))]
    if m.empty:
        return 'External/experimental replication: not found for this drug-subtype pair.'
    for c in ['replicated', 'validated', 'external_support', 'experimental_support']:
        if c in m.columns:
            val = bool(m[c].astype(bool).any())
            return 'External/experimental replication: SUPPORTED in supplied external table.' if val else 'External/experimental replication: present in external table but not marked supported.'
    return 'External/experimental replication: present in supplied external table.'

def make_discovery_statement(row: pd.Series, category: str, external_df: Optional[pd.DataFrame]=None) -> str:
    name = _clean_text_value(row.get('drug_name', '')) or f"DRUG_ID {int(row.get('drug_id'))}"
    subtype = _clean_text_value(row.get('tcga_desc', 'UNKNOWN')) or 'UNKNOWN'
    fdr = row.get('selectivity_fdr', np.nan)
    fdr_txt = f'FDR={float(fdr):.3g}' if np.isfinite(fdr) else 'FDR not estimable'
    fdr_status = 'signal survives strict FDR' if bool(row.get('fdr_pass', False)) else 'does not pass strict FDR'
    moa = _clean_text_value(row.get('moa_label', '')) or 'not available'
    pathway = _clean_text_value(row.get('pathway_label', '')) or 'not available'
    target = _clean_text_value(row.get('target_label', '')) or 'not available'
    ext = _external_replication_status(row, external_df)
    if bool(row.get('strong_candidate_pass', False)):
        status = 'strong computational candidate, not yet externally/experimentally validated'
    elif bool(row.get('strict_candidate_pass', False)):
        status = 'promising discovery candidate, not yet fully validated'
    else:
        status = 'hypothesis-generating signal only; not a strict discovery candidate'
    ln_ic50_txt = ''
    if np.isfinite(row.get('pred_ln_ic50_mean', np.nan)) and float(row.get('ln_ic50_available', 0.0)) > 0:
        ln_ic50_txt = f" predicted LN_IC50={row.get('pred_ln_ic50_mean', np.nan):.3f};"
    return f"[{category}] {name} is prioritized in {subtype}. Evidence: predicted AUC={row.get('pred_auc_mean', np.nan):.4f};{ln_ic50_txt} subtype rank={row.get('subtype_auc_rank', np.nan):.0f}; selectivity delta AUC={row.get('selectivity_delta_auc', np.nan):.4f}; {fdr_txt} ({fdr_status}); MoA={moa}; pathway={pathway}; target/TiC biology={target}; TiC support={row.get('tic_support_score', np.nan):.3f}; MoA support={row.get('moa_support_score', np.nan):.3f}; pathway support={row.get('pathway_support_score', np.nan):.3f}; discovery score={row.get('discovery_score', np.nan):.3f}. {ext} Final status: {status}."

def _strict_base_filter(cand: pd.DataFrame, auc_max: float=STRICT_DISCOVERY_AUC_MAX, rank_max: int=STRICT_DISCOVERY_RANK_MAX) -> pd.DataFrame:
    df = cand.copy()
    if 'valid_subtype' in df.columns:
        df = df[df['valid_subtype']]
    df = df[pd.to_numeric(df['pred_auc_mean'], errors='coerce').le(float(auc_max))]
    df = df[pd.to_numeric(df['subtype_auc_rank'], errors='coerce').le(float(rank_max))]
    return df

def _top10_for_category(cand: pd.DataFrame, category: str) -> pd.DataFrame:
    if cand.empty:
        return cand.copy()
    df = cand.copy()
    if category == 'Subtype-specific drug hit':
        df = _strict_base_filter(df, auc_max=STRICT_DISCOVERY_AUC_MAX, rank_max=STRICT_DISCOVERY_RANK_MAX)
        df = df[df['selectivity_delta_auc'].ge(STRICT_DISCOVERY_DELTA_MIN) & df['fdr_pass']]
        df = df.sort_values(['selectivity_delta_auc', 'subtype_auc_rank', 'pred_auc_mean'], ascending=[False, True, True])
    elif category == 'Top drug prioritization in a cancer subgroup':
        df = _strict_base_filter(df, auc_max=STRICT_DISCOVERY_AUC_MAX, rank_max=STRICT_DISCOVERY_RANK_STRONG)
        df = df.sort_values(['subtype_auc_rank', 'pred_auc_mean', 'fdr_pass', 'selectivity_delta_auc'], ascending=[True, True, False, False])
    elif category == 'MoA-supported vulnerability':
        df = _strict_base_filter(df, auc_max=0.75, rank_max=STRICT_DISCOVERY_RANK_MAX)
        df = df[df['has_moa_annotation'] & (df['moa_support_score'].ge(STRICT_DISCOVERY_BIOLOGY_MIN) | df['biology_support_score'].ge(STRICT_DISCOVERY_BIOLOGY_MIN))]
        df = df.sort_values(['moa_support_score', 'pred_auc_mean', 'subtype_auc_rank'], ascending=[False, True, True])
    elif category == 'Pathway-linked vulnerability':
        df = _strict_base_filter(df, auc_max=0.75, rank_max=STRICT_DISCOVERY_RANK_MAX)
        df = df[df['has_pathway_annotation'] & (df['pathway_support_score'].ge(STRICT_DISCOVERY_BIOLOGY_MIN) | df['biology_support_score'].ge(STRICT_DISCOVERY_BIOLOGY_MIN))]
        df = df.sort_values(['pathway_support_score', 'pred_auc_mean', 'subtype_auc_rank'], ascending=[False, True, True])
    elif category == 'TiC-supported mechanism':
        df = _strict_base_filter(df, auc_max=0.75, rank_max=STRICT_DISCOVERY_RANK_MAX)
        df = df[df['has_target_annotation'] & df['tic_support_score'].ge(STRICT_DISCOVERY_TIC_MIN)]
        df = df.sort_values(['tic_support_score', 'pred_auc_mean', 'subtype_auc_rank'], ascending=[False, True, True])
    elif category == 'Selectivity pattern across subtypes':
        df = _strict_base_filter(df, auc_max=0.75, rank_max=STRICT_DISCOVERY_RANK_MAX)
        df = df[df['selectivity_delta_auc'].ge(STRICT_DISCOVERY_DELTA_STRONG) & df['fdr_pass']]
        df = df.sort_values(['selectivity_delta_auc', 'selectivity_fdr', 'pred_auc_mean'], ascending=[False, True, True])
    elif category == 'Ranked therapeutic candidate':
        df = _strict_base_filter(df, auc_max=STRICT_DISCOVERY_AUC_STRONG, rank_max=STRICT_DISCOVERY_RANK_STRONG)
        df = df[df['fdr_pass'] & df['selectivity_delta_auc'].ge(STRICT_DISCOVERY_DELTA_MIN)]
        df = df.sort_values(['discovery_score', 'pred_auc_mean', 'subtype_auc_rank'], ascending=[False, True, True])
    elif category == 'Repurposing candidate':
        df = _strict_base_filter(df, auc_max=STRICT_DISCOVERY_AUC_MAX, rank_max=STRICT_DISCOVERY_RANK_MAX)
        df = df[df['has_moa_annotation'] | df['has_pathway_annotation'] | df['has_target_annotation']]
        df = df[df['selectivity_delta_auc'].ge(0.0)]
        df = df.sort_values(['discovery_score', 'pred_auc_mean'], ascending=[False, True])
    elif category == 'Hypothesis-generating biological signal':
        df = df[df['valid_subtype']]
        df = df[df['pred_auc_mean'].le(0.8) & df['subtype_auc_rank'].le(10)]
        df = df[df['has_moa_annotation'] | df['has_pathway_annotation'] | df['has_target_annotation'] | df['tic_support_score'].ge(STRICT_DISCOVERY_TIC_MIN)]
        df = df.sort_values(['discovery_score', 'pred_auc_mean'], ascending=[False, True])
    elif category == 'Promising discovery candidate, not yet fully validated':
        df = df[df['strict_candidate_pass']]
        df = df.sort_values(['strong_candidate_pass', 'discovery_score', 'pred_auc_mean'], ascending=[False, False, True])
    else:
        df = _strict_base_filter(df)
        df = df.sort_values('discovery_score', ascending=False)
    return df.head(10).copy()

def print_discovery_top10_report(result: Dict[str, Any], all_df: Optional[pd.DataFrame]=None, external_df: Optional[pd.DataFrame]=None, output_dir: Optional[str]=None, top_n: int=10) -> Dict[str, pd.DataFrame]:
    pred_df = _combine_test_predictions_from_result(result)
    if pred_df.empty:
        print('\n[DISCOVERY REPORT] No test predictions found; discovery report skipped.')
        return {}
    cand = build_subtype_candidate_table(pred_df, all_df=all_df)
    if cand.empty:
        print('\n[DISCOVERY REPORT] Candidate table is empty; discovery report skipped.')
        return {}
    categories = ['Subtype-specific drug hit', 'Top drug prioritization in a cancer subgroup', 'MoA-supported vulnerability', 'Pathway-linked vulnerability', 'TiC-supported mechanism', 'Selectivity pattern across subtypes', 'Ranked therapeutic candidate', 'Repurposing candidate', 'Hypothesis-generating biological signal', 'Promising discovery candidate, not yet fully validated']
    reports: Dict[str, pd.DataFrame] = {'all_candidates': cand}
    if output_dir is None:
        output_dir = result.get('output_dir', 'druxr_outputs/discovery_report')
    report_dir = os.path.join(output_dir, 'discovery_report')
    ensure_dir(report_dir)
    cand.to_csv(os.path.join(report_dir, 'all_discovery_candidates.csv'), index=False)
    print('\n' + '=' * 96)
    print('DISCOVERY-STYLE REPORT — computational candidates only')
    print('Each evidence category prints TOP 10 candidates AFTER strict response-first filters.')
    print('Rules: low AUC + top subtype rank first; biology/TiC only support candidates, they cannot rescue high-AUC hits.')
    print('External/experimental validation is reported only if external_df is supplied.')
    print('=' * 96)
    combined_rows = []
    for category in categories:
        top = _top10_for_category(cand, category).head(int(top_n)).copy()
        top['discovery_category'] = category
        top['discovery_statement'] = [make_discovery_statement(r, category, external_df) for _, r in top.iterrows()]
        reports[category] = top
        top.to_csv(os.path.join(report_dir, category.lower().replace(' ', '_').replace('/', '_').replace(',', '') + '_top10.csv'), index=False)
        combined_rows.append(top)
        print('\n' + '-' * 96)
        print(f'TOP {min(int(top_n), len(top))}: {category}')
        print('-' * 96)
        if top.empty:
            print('No candidates passed the strict filters for this category.')
        else:
            for i, (_, row) in enumerate(top.iterrows(), start=1):
                print(f'{i:02d}. ' + row['discovery_statement'])
    if combined_rows:
        combined = pd.concat(combined_rows, ignore_index=True)
        combined.to_csv(os.path.join(report_dir, 'discovery_top10_all_categories.csv'), index=False)
        reports['top10_all_categories'] = combined
    print('\nDiscovery report saved under:', report_dir)
    print("Recommended wording: use 'strong/promising computational candidate' only for rows passing strict filters; otherwise 'hypothesis-generating signal only'.")
    return reports

def _eval_one_epoch_dual_response_base(model, loader, device: str, cfg: ConfigDruXRV5):
    model.eval()
    running: Dict[str, float] = {}
    y_tic, y_tic_hat, tic_mask_all = ([], [], [])
    moa_true, moa_score = ([], [])
    sel_true, sel_score, sel_mask_all = ([], [], [])
    pathway_true, pathway_score = ([], [])
    pred_rows = []
    for batch in loader:
        batch = move_batch_to_device(batch, device)
        out, losses = compute_losses(model, batch, cfg, aux_scale=1.0)
        for k, v in losses.items():
            running[k] = running.get(k, 0.0) + float(v.detach().cpu())
        y_tic.append(batch['tic_target'].detach().cpu().view(-1).numpy())
        y_tic_hat.append(out['tic'].detach().cpu().view(-1).numpy())
        tic_mask_all.append(batch['tic_mask'].detach().cpu().view(-1).numpy())
        if 'moa' in out and batch['moa_target'].numel() > 0:
            moa_true.append(batch['moa_target'].detach().cpu().numpy())
            moa_score.append(torch.sigmoid(out['moa']).detach().cpu().numpy())
        if 'selectivity' in out and batch['sel_target'].numel() > 0:
            sel_true.append(batch['sel_target'].detach().cpu().numpy())
            sel_score.append(torch.sigmoid(out['selectivity']).detach().cpu().numpy())
            sel_mask_all.append(batch['sel_mask'].detach().cpu().view(-1).numpy())
        if 'pathway' in out and batch['pathway_target'].numel() > 0:
            pathway_true.append(batch['pathway_target'].detach().cpu().numpy())
            pathway_score.append(torch.sigmoid(out['pathway']).detach().cpu().numpy())
        for i in range(len(batch['cell_line'])):
            moa_support = np.nan
            pathway_support = np.nan
            selectivity_support = np.nan
            if 'moa' in out:
                moa_support = float(torch.sigmoid(out['moa'][i]).detach().cpu().max().item())
            if 'pathway' in out:
                pathway_support = float(torch.sigmoid(out['pathway'][i]).detach().cpu().max().item())
            if 'selectivity' in out:
                selectivity_support = float(torch.sigmoid(out['selectivity'][i]).detach().cpu().max().item())
            pred_rows.append({'drug_id': int(batch['drug_id'][i].detach().cpu().item()), 'drug_name': batch.get('drug_name', [str(int(batch['drug_id'][i].detach().cpu().item()))] * len(batch['cell_line']))[i], 'cell_line': batch['cell_line'][i], 'tcga_desc': batch['tcga_desc'][i], 'response_true': float(batch['response_target'][i].detach().cpu().item()), 'response_pred': float(out['response'][i].detach().cpu().item()), 'ln_ic50_true': float(batch['ln_ic50_target'][i].detach().cpu().item()), 'ln_ic50_pred': float(out['ln_ic50'][i].detach().cpu().item()), 'ln_ic50_mask': float(batch['ln_ic50_mask'][i].detach().cpu().item()), 'rank_pred': float(out['rank'][i].detach().cpu().item()), 'tic_true': float(batch['tic_target'][i].detach().cpu().item()), 'tic_pred': float(out['tic'][i].detach().cpu().item()), 'moa_pred_max': moa_support, 'pathway_pred_max': pathway_support, 'selectivity_pred_max': selectivity_support})
    running = {k: v / max(len(loader), 1) for k, v in running.items()}
    pred_df = pd.DataFrame(pred_rows)
    running.update(response_metrics_from_df(pred_df, cfg.topk_values))
    running.update(ln_ic50_metrics_from_df(pred_df, cfg.topk_values))
    if y_tic:
        yt, yp, m = (np.concatenate(y_tic), np.concatenate(y_tic_hat), np.concatenate(tic_mask_all).astype(bool))
        if m.sum() > 0:
            running['tic_rmse'] = float(np.sqrt(np.mean((yt[m] - yp[m]) ** 2)))
            running['tic_spearman'] = float(spearmanr(yt[m], yp[m]).correlation) if m.sum() > 1 else np.nan
        else:
            running['tic_rmse'] = np.nan
            running['tic_spearman'] = np.nan
    if moa_true:
        mm = compute_multilabel_metrics(np.concatenate(moa_true), np.concatenate(moa_score))
        running.update({f'moa_{k}': v for k, v in mm.items()})
    if sel_true:
        mask = np.concatenate(sel_mask_all).astype(bool) if sel_mask_all else None
        if mask is not None and mask.sum() > 0:
            st = np.concatenate(sel_true)[mask]
            ss = np.concatenate(sel_score)[mask]
            sm = compute_multilabel_metrics(st, ss)
            running.update({f'selectivity_{k}': v for k, v in sm.items()})
        else:
            running['selectivity_auroc_macro'] = np.nan
            running['selectivity_auprc_macro'] = np.nan
    if pathway_true:
        pt, ps = (np.concatenate(pathway_true), np.concatenate(pathway_score))
        pm = compute_multilabel_metrics(pt, ps)
        running.update({f'pathway_{k}': v for k, v in pm.items()})
        if pt.shape[1] > 0:
            running['pathway_accuracy'] = float((np.argmax(pt, axis=1) == np.argmax(ps, axis=1)).mean())
    return (running, pred_df)

def _finite_mean(values: Iterable[float]) -> float:
    vals = [float(v) for v in values if v is not None and np.isfinite(float(v))]
    return float(np.mean(vals)) if vals else np.nan

def eval_one_epoch(model, loader, device: str, cfg: ConfigDruXRV5):
    metrics, pred_df = _eval_one_epoch_dual_response_base(model, loader, device, cfg)
    metrics['biology_composite'] = _finite_mean([metrics.get('moa_auroc_macro', np.nan), metrics.get('pathway_auroc_macro', np.nan), metrics.get('moa_auprc_macro', np.nan), metrics.get('pathway_auprc_macro', np.nan)])
    metrics['selectivity_composite'] = _finite_mean([metrics.get('selectivity_auroc_macro', np.nan), metrics.get('selectivity_auprc_macro', np.nan)])
    metrics['response_auc_ranking_composite'] = metrics.get('response_discovery_composite', metrics.get('response_composite', np.nan))
    metrics['ln_ic50_ranking_composite'] = metrics.get('ln_ic50_discovery_composite', _finite_mean([metrics.get('ln_ic50_spearman', np.nan), metrics.get('ln_ic50_spearman_drug_macro', np.nan)]))
    return (metrics, pred_df)
SPECIALIST_SPECS = {'response_auc': {'purpose': 'Primary sensitivity ranking', 'heads_outputs': 'AUC head + rank head + subtype/drug listwise rank', 'monitor_metric': 'response_auc_ranking_composite', 'monitor_mode': 'max', 'weights': {'lambda_response': 2.0, 'lambda_ln_ic50': 0.05, 'lambda_rank': 0.2, 'lambda_response_pair_rank': 0.3, 'lambda_response_global_rank': 0.03, 'lambda_response_subtype_listwise': 0.1, 'lambda_response_drug_listwise': 0.1, 'lambda_ln_ic50_pair_rank': 0.02, 'lambda_ln_ic50_global_rank': 0.0, 'lambda_ln_ic50_subtype_listwise': 0.0, 'lambda_ln_ic50_drug_listwise': 0.0, 'lambda_tic': 0.03, 'lambda_moa': 0.0, 'lambda_pathway': 0.0, 'lambda_selectivity': 0.02, 'lambda_domain_adv': 0.0}}, 'response_ln_ic50': {'purpose': 'IC50 confirmation/support', 'heads_outputs': 'LN_IC50 head + LN_IC50 listwise rank', 'monitor_metric': 'ln_ic50_spearman_drug_macro', 'monitor_mode': 'max', 'weights': {'lambda_response': 0.1, 'lambda_ln_ic50': 2.0, 'lambda_rank': 0.0, 'lambda_response_pair_rank': 0.02, 'lambda_response_global_rank': 0.0, 'lambda_response_subtype_listwise': 0.0, 'lambda_response_drug_listwise': 0.0, 'lambda_ln_ic50_pair_rank': 0.3, 'lambda_ln_ic50_global_rank': 0.03, 'lambda_ln_ic50_subtype_listwise': 0.08, 'lambda_ln_ic50_drug_listwise': 0.08, 'lambda_tic': 0.0, 'lambda_moa': 0.0, 'lambda_pathway': 0.0, 'lambda_selectivity': 0.0, 'lambda_domain_adv': 0.0}}, 'tic': {'purpose': 'Target-neighborhood mechanism', 'heads_outputs': 'TiC head', 'monitor_metric': 'tic_spearman', 'monitor_mode': 'max', 'weights': {'lambda_response': 0.15, 'lambda_ln_ic50': 0.0, 'lambda_rank': 0.0, 'lambda_response_pair_rank': 0.02, 'lambda_response_global_rank': 0.0, 'lambda_response_subtype_listwise': 0.0, 'lambda_response_drug_listwise': 0.0, 'lambda_ln_ic50_pair_rank': 0.0, 'lambda_ln_ic50_global_rank': 0.0, 'lambda_ln_ic50_subtype_listwise': 0.0, 'lambda_ln_ic50_drug_listwise': 0.0, 'lambda_tic': 2.0, 'lambda_moa': 0.0, 'lambda_pathway': 0.0, 'lambda_selectivity': 0.0, 'lambda_domain_adv': 0.0}}, 'biology': {'purpose': 'Biological mechanism support', 'heads_outputs': 'MoA head + pathway head', 'monitor_metric': 'biology_composite', 'monitor_mode': 'max', 'weights': {'lambda_response': 0.15, 'lambda_ln_ic50': 0.0, 'lambda_rank': 0.0, 'lambda_response_pair_rank': 0.02, 'lambda_response_global_rank': 0.0, 'lambda_response_subtype_listwise': 0.0, 'lambda_response_drug_listwise': 0.0, 'lambda_ln_ic50_pair_rank': 0.0, 'lambda_ln_ic50_global_rank': 0.0, 'lambda_ln_ic50_subtype_listwise': 0.0, 'lambda_ln_ic50_drug_listwise': 0.0, 'lambda_tic': 0.0, 'lambda_moa': 1.0, 'lambda_pathway': 1.0, 'lambda_selectivity': 0.0, 'lambda_domain_adv': 0.0}}, 'selectivity': {'purpose': 'Subtype-specific vulnerability', 'heads_outputs': 'Selectivity head + rank/selectivity score', 'monitor_metric': 'response_auc_ranking_composite', 'monitor_mode': 'max', 'weights': {'lambda_response': 0.35, 'lambda_ln_ic50': 0.02, 'lambda_rank': 0.25, 'lambda_response_pair_rank': 0.25, 'lambda_response_global_rank': 0.02, 'lambda_response_subtype_listwise': 0.08, 'lambda_response_drug_listwise': 0.12, 'lambda_ln_ic50_pair_rank': 0.02, 'lambda_ln_ic50_global_rank': 0.0, 'lambda_ln_ic50_subtype_listwise': 0.0, 'lambda_ln_ic50_drug_listwise': 0.0, 'lambda_tic': 0.0, 'lambda_moa': 0.0, 'lambda_pathway': 0.0, 'lambda_selectivity': 2.0, 'lambda_domain_adv': 0.0}}}

def specialist_design_table() -> pd.DataFrame:
    rows = []
    for name, spec in SPECIALIST_SPECS.items():
        rows.append({'Specialist': name, 'Heads / outputs': spec['heads_outputs'], 'Purpose': spec['purpose'], 'Monitor metric': spec['monitor_metric'], 'Monitor mode': spec['monitor_mode']})
    return pd.DataFrame(rows)

def _print_df_block(title: str, df: pd.DataFrame, max_rows: int=50) -> None:
    print('\n' + '=' * 100)
    print(title)
    print('=' * 100)
    if df is None or df.empty:
        print('[empty]')
    else:
        with pd.option_context('display.max_rows', max_rows, 'display.max_columns', None, 'display.width', 240):
            print(df)

def _make_specialist_cfg(base_cfg: ConfigDruXRV5, specialist: str, root_dir: str) -> ConfigDruXRV5:
    if specialist not in SPECIALIST_SPECS:
        raise ValueError(f'Unknown specialist: {specialist}. Choose from {list(SPECIALIST_SPECS)}')
    cfg = ConfigDruXRV5(**asdict(base_cfg))
    for k, v in vars(base_cfg).items():
        if not hasattr(cfg, k):
            setattr(cfg, k, v)
    _ensure_v5_config_defaults(cfg)
    spec = SPECIALIST_SPECS[specialist]
    cfg.output_dir = root_dir
    cfg.run_name = specialist
    cfg.monitor_metric = spec['monitor_metric']
    cfg.monitor_mode = spec['monitor_mode']
    for k, v in spec['weights'].items():
        setattr(cfg, k, float(v))
    cfg.specialist_name = specialist
    return cfg

def _extract_summary_row(result: Dict[str, Any], specialist: str) -> Dict[str, Any]:
    sd = result.get('summary_dict', {}) if isinstance(result, dict) else {}
    row = {'specialist': specialist, 'purpose': SPECIALIST_SPECS[specialist]['purpose'], 'heads_outputs': SPECIALIST_SPECS[specialist]['heads_outputs'], 'monitor_metric': SPECIALIST_SPECS[specialist]['monitor_metric'], 'monitor_mode': SPECIALIST_SPECS[specialist]['monitor_mode'], 'n_seeds': sd.get('n_seeds', np.nan), 'n_split_seeds': sd.get('n_split_seeds', np.nan), 'n_runs': sd.get('n_runs', np.nan), 'test_auc_rmse': sd.get('test_response_rmse_mean', np.nan), 'test_auc_spearman': sd.get('test_response_spearman_mean', np.nan), 'test_auc_drug_macro_spearman': sd.get('test_response_spearman_drug_macro_mean', np.nan), 'test_ln_ic50_rmse': sd.get('test_ln_ic50_rmse_mean', np.nan), 'test_ln_ic50_spearman': sd.get('test_ln_ic50_spearman_mean', np.nan), 'test_ln_ic50_drug_macro_spearman': sd.get('test_ln_ic50_spearman_drug_macro_mean', np.nan), 'test_tic_spearman': sd.get('test_tic_spearman_mean', np.nan), 'test_moa_auroc': sd.get('test_moa_auroc_macro_mean', np.nan), 'test_moa_auprc': sd.get('test_moa_auprc_macro_mean', np.nan), 'test_pathway_auroc': sd.get('test_pathway_auroc_macro_mean', np.nan), 'test_pathway_auprc': sd.get('test_pathway_auprc_macro_mean', np.nan), 'test_selectivity_auroc': sd.get('test_posthoc_selectivity_auroc_mean', sd.get('test_selectivity_auroc_macro_mean', np.nan)), 'test_selectivity_auprc': sd.get('test_posthoc_selectivity_auprc_mean', sd.get('test_selectivity_auprc_macro_mean', np.nan)), 'test_selectivity_spearman': sd.get('test_posthoc_selectivity_spearman_mean', np.nan), 'test_top10_precision': sd.get('test_top10_precision_mean', np.nan), 'test_top10_ndcg': sd.get('test_top10_ndcg_mean', np.nan), 'test_top10_subtype_drug_ndcg': sd.get('test_top10_subtype_drug_ndcg_macro_mean', np.nan), 'test_top10_drug_subtype_ndcg': sd.get('test_top10_drug_subtype_ndcg_macro_mean', np.nan)}
    return row

def _preds_by_seed(result: Dict[str, Any], split: str='test') -> Dict[Tuple[int, int], pd.DataFrame]:
    out: Dict[Tuple[int, int], pd.DataFrame] = {}
    key = 'val_predictions' if split == 'val' else 'test_predictions'
    for run_key, seed_result in result.get('seed_results', {}).items():
        pred = seed_result.get(key)
        if isinstance(pred, pd.DataFrame) and (not pred.empty):
            tmp = pred.copy()
            split_seed = int(seed_result.get('split_seed', tmp['split_seed'].iloc[0] if 'split_seed' in tmp.columns else 1))
            model_seed = int(seed_result.get('model_seed', tmp['model_seed'].iloc[0] if 'model_seed' in tmp.columns else tmp['seed'].iloc[0] if 'seed' in tmp.columns else 1))
            out[split_seed, model_seed] = tmp
    return out

def _merge_specialist_predictions_for_seed(specialist_results: Dict[str, Dict[str, Any]], seed: Tuple[int, int], model_seed: Optional[int]=None, split: str='test') -> Optional[pd.DataFrame]:
    if model_seed is not None and (not isinstance(seed, tuple)):
        run_key = (int(seed), int(model_seed))
    else:
        run_key = tuple(seed) if isinstance(seed, tuple) else (1, int(seed))
    split_seed, model_seed = (int(run_key[0]), int(run_key[1]))
    keys = ['drug_id', 'cell_line', 'tcga_desc']
    response_frames = _preds_by_seed(specialist_results.get('response_auc', {}), split=split)
    if run_key not in response_frames:
        return None
    base = response_frames[run_key].copy()
    for k in keys:
        if k not in base.columns:
            if k == 'tcga_desc':
                base[k] = 'UNKNOWN'
            else:
                return None
    keep = keys + (['drug_name'] if 'drug_name' in base.columns else []) + ['response_true', 'response_pred', 'rank_pred', 'tic_true', 'tic_pred', 'ln_ic50_true', 'ln_ic50_pred', 'ln_ic50_mask', 'moa_pred_max', 'pathway_pred_max', 'selectivity_pred_max']
    keep = [c for c in keep if c in base.columns]
    combo = base[keep].copy()
    combo = combo.rename(columns={'response_pred': 'response_pred_auc_specialist', 'rank_pred': 'rank_pred_auc_specialist', 'tic_pred': 'tic_pred_auc_specialist', 'ln_ic50_pred': 'ln_ic50_pred_auc_specialist', 'moa_pred_max': 'moa_pred_max_auc_specialist', 'pathway_pred_max': 'pathway_pred_max_auc_specialist', 'selectivity_pred_max': 'selectivity_pred_max_auc_specialist'})
    combo['response_pred'] = combo['response_pred_auc_specialist']
    if 'rank_pred_auc_specialist' in combo.columns:
        combo['rank_pred'] = combo['rank_pred_auc_specialist']
    if 'tic_pred_auc_specialist' in combo.columns:
        combo['tic_pred'] = combo['tic_pred_auc_specialist']
    if 'ln_ic50_pred_auc_specialist' in combo.columns:
        combo['ln_ic50_pred'] = combo['ln_ic50_pred_auc_specialist']

    def merge_col_from_specialist(specialist: str, src_col: str, dst_col: str):
        nonlocal combo
        frames = _preds_by_seed(specialist_results.get(specialist, {}), split=split)
        if run_key not in frames:
            combo[dst_col] = np.nan
            return
        p = frames[run_key].copy()
        if not all((k in p.columns for k in keys)) or src_col not in p.columns:
            combo[dst_col] = np.nan
            return
        add = p[keys + [src_col]].rename(columns={src_col: dst_col})
        combo = combo.merge(add, on=keys, how='left')
    merge_col_from_specialist('response_ln_ic50', 'ln_ic50_pred', 'ln_ic50_pred_ln_ic50_specialist')
    merge_col_from_specialist('tic', 'tic_pred', 'tic_pred_tic_specialist')
    merge_col_from_specialist('selectivity', 'rank_pred', 'rank_pred_selectivity_specialist')
    merge_col_from_specialist('selectivity', 'response_pred', 'response_pred_selectivity_specialist')
    merge_col_from_specialist('selectivity', 'selectivity_pred_max', 'selectivity_pred_max_selectivity_specialist')
    merge_col_from_specialist('biology', 'rank_pred', 'rank_pred_biology_specialist')
    merge_col_from_specialist('biology', 'moa_pred_max', 'moa_pred_max_biology_specialist')
    merge_col_from_specialist('biology', 'pathway_pred_max', 'pathway_pred_max_biology_specialist')
    if 'response_pred_selectivity_specialist' in combo.columns and 'response_pred_auc_specialist' in combo.columns:
        a = pd.to_numeric(combo['response_pred_auc_specialist'], errors='coerce')
        b = pd.to_numeric(combo['response_pred_selectivity_specialist'], errors='coerce')
        combo['response_pred'] = (0.8 * a + 0.2 * b).where(a.notna() & b.notna(), a.combine_first(b))
    if 'ln_ic50_pred_ln_ic50_specialist' in combo.columns:
        combo['ln_ic50_pred'] = combo['ln_ic50_pred_ln_ic50_specialist'].combine_first(combo.get('ln_ic50_pred', pd.Series(np.nan, index=combo.index)))
    if 'tic_pred_tic_specialist' in combo.columns:
        combo['tic_pred'] = combo['tic_pred_tic_specialist'].combine_first(combo.get('tic_pred', pd.Series(np.nan, index=combo.index)))
    if 'moa_pred_max_biology_specialist' in combo.columns:
        combo['moa_pred_max'] = combo['moa_pred_max_biology_specialist'].combine_first(combo.get('moa_pred_max_auc_specialist', pd.Series(np.nan, index=combo.index)))
    if 'pathway_pred_max_biology_specialist' in combo.columns:
        combo['pathway_pred_max'] = combo['pathway_pred_max_biology_specialist'].combine_first(combo.get('pathway_pred_max_auc_specialist', pd.Series(np.nan, index=combo.index)))
    if 'selectivity_pred_max_selectivity_specialist' in combo.columns:
        combo['selectivity_pred_max'] = combo['selectivity_pred_max_selectivity_specialist'].combine_first(combo.get('selectivity_pred_max_auc_specialist', pd.Series(np.nan, index=combo.index)))
    rank_cols = [c for c in ['rank_pred_auc_specialist', 'rank_pred_selectivity_specialist', 'rank_pred_biology_specialist'] if c in combo.columns]
    if rank_cols:
        combo['rank_pred'] = combo[rank_cols].mean(axis=1, skipna=True)
    combo['split'] = split
    combo['split_seed'] = split_seed
    combo['seed'] = model_seed
    combo['model_seed'] = model_seed
    combo['run_key'] = f'split{split_seed}_seed{model_seed}'
    return combo

def build_multispecialist_combined_predictions(specialist_results):
    run_keys = sorted(set().union(*[set(_preds_by_seed(r, split='test').keys()) for r in specialist_results.values() if isinstance(r, dict)]))
    val_frames, test_frames = ([], [])
    for run_key in run_keys:
        val_merged = _merge_specialist_predictions_for_seed(specialist_results, run_key, split='val')
        test_merged = _merge_specialist_predictions_for_seed(specialist_results, run_key, split='test')
        if val_merged is not None and (not val_merged.empty):
            val_frames.append(val_merged)
        if test_merged is not None and (not test_merged.empty):
            test_frames.append(test_merged)
    val_all = pd.concat(val_frames, ignore_index=True) if val_frames else pd.DataFrame()
    test_all = pd.concat(test_frames, ignore_index=True) if test_frames else pd.DataFrame()
    cfg = None
    try:
        for res in specialist_results.values():
            if isinstance(res, dict) and res.get('config', None) is not None:
                cfg = res.get('config')
                break
    except Exception:
        cfg = None
    val_all, test_all, fusion_info = _apply_validation_fusion(val_all, test_all, cfg=cfg)
    build_multispecialist_combined_predictions.last_val_predictions = val_all
    build_multispecialist_combined_predictions.fusion_info = fusion_info
    return test_all

def print_multispecialist_metric_report(specialist_results: Dict[str, Dict[str, Any]], output_dir: str) -> pd.DataFrame:
    rows = [_extract_summary_row(res, sp) for sp, res in specialist_results.items()]
    overview = pd.DataFrame(rows)
    ensure_dir(output_dir)
    overview.to_csv(os.path.join(output_dir, 'multispecialist_overview_metrics.csv'), index=False)
    _print_df_block('MULTI-SPECIALIST DESIGN', specialist_design_table())
    _print_df_block('MULTI-SPECIALIST OVERVIEW METRICS', overview)
    for sp, res in specialist_results.items():
        summ = res.get('summary')
        if isinstance(summ, pd.DataFrame):
            summ.to_csv(os.path.join(output_dir, f'{sp}_summary.csv'), index=False)
            _print_df_block(f'FULL SUMMARY — {sp}', summ)
        metrics = res.get('metrics')
        if isinstance(metrics, pd.DataFrame):
            metrics.to_csv(os.path.join(output_dir, f'{sp}_seed_metrics.csv'), index=False)
            _print_df_block(f'PER-SEED METRICS — {sp}', metrics, max_rows=200)
    return overview

def _expr_stats(vals: np.ndarray) -> Dict[str, float]:
    vals = np.asarray(vals, dtype=np.float32)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return {'mean': 0.0, 'std': 0.0, 'q25': 0.0, 'q75': 0.0, 'max': 0.0, 'min': 0.0, 'frac_high': 0.0}
    return {'mean': float(vals.mean()), 'std': float(vals.std(ddof=0)), 'q25': float(np.quantile(vals, 0.25)), 'q75': float(np.quantile(vals, 0.75)), 'max': float(vals.max()), 'min': float(vals.min()), 'frac_high': float((vals > 0.5).mean())}

def informative_pairwise_rank_loss(scores: torch.Tensor, targets: torch.Tensor, groups: torch.Tensor, mask: Optional[torch.Tensor]=None, min_delta: float=0.05, max_pairs: int=4096) -> torch.Tensor:
    scores, targets, groups = (scores.view(-1), targets.view(-1), groups.view(-1))
    if mask is not None:
        keep = mask.view(-1) > 0
        scores, targets, groups = (scores[keep], targets[keep], groups[keep])
    if scores.numel() < 2:
        return torch.tensor(0.0, device=scores.device)
    losses = []
    for g in torch.unique(groups):
        idx = torch.where(groups == g)[0]
        if idx.numel() < 2:
            continue
        s, y = (scores[idx], targets[idx])
        dy = y.unsqueeze(1) - y.unsqueeze(0)
        ds = s.unsqueeze(1) - s.unsqueeze(0)
        valid = torch.triu(torch.ones_like(dy, dtype=torch.bool), diagonal=1) & (dy.abs() >= float(min_delta))
        if valid.sum() == 0:
            continue
        sign = torch.sign(dy[valid])
        dscore = ds[valid]
        if max_pairs is not None and valid.sum().item() > int(max_pairs):
            perm = torch.randperm(valid.sum().item(), device=scores.device)[:int(max_pairs)]
            sign = sign[perm]
            dscore = dscore[perm]
        losses.append(F.softplus(-sign * dscore).mean())
    return torch.stack(losses).mean() if losses else torch.tensor(0.0, device=scores.device)

def group_listnet_rank_loss(scores: torch.Tensor, targets: torch.Tensor, groups: torch.Tensor, mask: Optional[torch.Tensor]=None, temperature: float=0.15, min_delta: float=0.05) -> torch.Tensor:
    scores, targets, groups = (scores.view(-1), targets.view(-1), groups.view(-1))
    if mask is not None:
        keep = mask.view(-1) > 0
        scores, targets, groups = (scores[keep], targets[keep], groups[keep])
    if scores.numel() < 3:
        return torch.tensor(0.0, device=scores.device)
    losses = []
    tau = max(float(temperature), 0.001)
    for g in torch.unique(groups):
        idx = torch.where(groups == g)[0]
        if idx.numel() < 3:
            continue
        s, y = (scores[idx], targets[idx])
        if (torch.max(y) - torch.min(y)).detach().item() < float(min_delta):
            continue
        target_prob = torch.softmax((y - y.mean()).detach() / tau, dim=0)
        pred_log_prob = torch.log_softmax((s - s.mean()) / tau, dim=0)
        losses.append(-(target_prob * pred_log_prob).sum())
    return torch.stack(losses).mean() if losses else torch.tensor(0.0, device=scores.device)

def _valid_group_label_for_metrics(x: Any) -> bool:
    s = str(x).strip().upper()
    return s not in {'', 'NAN', 'NONE', 'UNKNOWN', 'UNCLASSIFIED', '<NA>'}

def _grouped_topk_macro(df: pd.DataFrame, group_col: str, item_col: str, true_col: str, pred_col: str, k_values: Iterable[int], prefix: str) -> Dict[str, float]:
    out: Dict[str, float] = {}
    if df.empty or not {group_col, item_col, true_col, pred_col}.issubset(df.columns):
        return out
    agg = df[df[group_col].map(_valid_group_label_for_metrics)].groupby([group_col, item_col], as_index=False).agg(true_mean=(true_col, 'mean'), pred_mean=(pred_col, 'mean'), n=(true_col, 'size'))
    for k in k_values:
        ps, ns = ([], [])
        for _, sub in agg.groupby(group_col):
            if len(sub) < 2:
                continue
            kk = min(int(k), len(sub))
            true_o = np.argsort(sub['true_mean'].astype(float).values)
            pred_o = np.argsort(sub['pred_mean'].astype(float).values)
            p, _, nd = _binary_topk_stats(true_o, pred_o, kk)
            if np.isfinite(p):
                ps.append(p)
            if np.isfinite(nd):
                ns.append(nd)
        out[f'top{k}_{prefix}_precision_macro'] = float(np.mean(ps)) if ps else np.nan
        out[f'top{k}_{prefix}_ndcg_macro'] = float(np.mean(ns)) if ns else np.nan
    return out

def posthoc_selectivity_metrics_from_df(pred_df: pd.DataFrame, true_col: str='response_true', pred_col: str='response_pred', delta_threshold: float=0.05, min_cells: int=2) -> Dict[str, float]:
    out = {'posthoc_selectivity_spearman': np.nan, 'posthoc_selectivity_auroc': np.nan, 'posthoc_selectivity_auprc': np.nan}
    need = {'drug_id', 'tcga_desc', true_col, pred_col}
    if pred_df.empty or not need.issubset(pred_df.columns):
        return out
    df = pred_df.copy()
    df = df[df['tcga_desc'].map(_valid_group_label_for_metrics)]
    if df.empty:
        return out
    agg = df.groupby(['drug_id', 'tcga_desc'], as_index=False).agg(true_sub=(true_col, 'mean'), pred_sub=(pred_col, 'mean'), n_cells=(true_col, 'size'))
    agg = agg[agg['n_cells'] >= int(min_cells)]
    if agg.empty:
        return out
    true_global = df.groupby('drug_id')[true_col].mean().rename('true_global').reset_index()
    pred_global = df.groupby('drug_id')[pred_col].mean().rename('pred_global').reset_index()
    agg = agg.merge(true_global, on='drug_id', how='left').merge(pred_global, on='drug_id', how='left')
    agg['true_selectivity_delta'] = agg['true_global'] - agg['true_sub']
    agg['pred_selectivity_delta'] = agg['pred_global'] - agg['pred_sub']
    if len(agg) > 2 and agg['true_selectivity_delta'].nunique() > 1 and (agg['pred_selectivity_delta'].nunique() > 1):
        out['posthoc_selectivity_spearman'] = float(spearmanr(agg['true_selectivity_delta'], agg['pred_selectivity_delta']).correlation)
    y = (agg['true_selectivity_delta'] >= float(delta_threshold)).astype(int).values
    s = agg['pred_selectivity_delta'].astype(float).values
    if len(np.unique(y)) > 1:
        try:
            out['posthoc_selectivity_auroc'] = float(roc_auc_score(y, s))
        except Exception:
            pass
        try:
            out['posthoc_selectivity_auprc'] = float(average_precision_score(y, s))
        except Exception:
            pass
    return out

def _fusion_feature_columns(df: pd.DataFrame) -> List[str]:
    candidates = ['response_pred_auc_specialist', 'response_pred_selectivity_specialist', 'rank_pred_auc_specialist', 'rank_pred_selectivity_specialist', 'rank_pred_biology_specialist', 'ln_ic50_pred_auc_specialist', 'ln_ic50_pred_ln_ic50_specialist', 'tic_pred_auc_specialist', 'tic_pred_tic_specialist', 'moa_pred_max_auc_specialist', 'moa_pred_max_biology_specialist', 'pathway_pred_max_auc_specialist', 'pathway_pred_max_biology_specialist', 'selectivity_pred_max_auc_specialist', 'selectivity_pred_max_selectivity_specialist', 'rank_pred', 'tic_pred', 'moa_pred_max', 'pathway_pred_max', 'selectivity_pred_max']
    return [c for c in candidates if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]

def _prepare_fusion_matrix(train_df: pd.DataFrame, apply_df: pd.DataFrame, feature_cols: List[str]):
    train_x = train_df[feature_cols].replace([np.inf, -np.inf], np.nan).copy()
    med = train_x.median(numeric_only=True).fillna(0.0)
    train_x = train_x.fillna(med).astype(float).values
    apply_x = apply_df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(med).astype(float).values
    return (train_x, apply_x, med.to_dict())

def _apply_validation_fusion(val_df: pd.DataFrame, test_df: pd.DataFrame, cfg: Optional[ConfigDruXRV5]=None):
    if cfg is None:
        cfg = CFG_DRUXR_V5
    _ensure_v5_config_defaults(cfg)
    info: Dict[str, Any] = {'used_response_fusion': False, 'used_ln_ic50_fusion': False}
    if Ridge is None or not bool(getattr(cfg, 'use_specialist_fusion', True)) or val_df.empty or test_df.empty:
        return (val_df, test_df, info)
    feature_cols = _fusion_feature_columns(val_df)
    feature_cols = [c for c in feature_cols if c in test_df.columns]
    if len(feature_cols) < 2:
        info['reason'] = 'too_few_features'
        return (val_df, test_df, info)
    min_rows = int(getattr(cfg, 'fusion_min_rows', 40))
    val_clean = val_df.dropna(subset=['response_true', 'response_pred']).copy()
    if len(val_clean) >= min_rows:
        Xv, Xt, med = _prepare_fusion_matrix(val_clean, test_df, feature_cols)
        yv = val_clean['response_true'].astype(float).values
        model = Ridge(alpha=float(getattr(cfg, 'fusion_alpha', 1.0)))
        model.fit(Xv, yv)
        val_fused = model.predict(Xv)
        test_fused = model.predict(Xt)
        if bool(getattr(cfg, 'clip_auc_predictions', True)):
            val_fused = np.clip(val_fused, 0.0, 1.0)
            test_fused = np.clip(test_fused, 0.0, 1.0)
        base_metrics = response_metrics_from_df(val_clean.assign(response_pred=val_clean['response_pred'].values))
        fused_metrics = response_metrics_from_df(val_clean.assign(response_pred=val_fused))
        base_score = base_metrics.get('response_discovery_composite', -np.inf)
        fused_score = fused_metrics.get('response_discovery_composite', -np.inf)
        info['response_base_val_score'] = float(base_score) if np.isfinite(base_score) else None
        info['response_fused_val_score'] = float(fused_score) if np.isfinite(fused_score) else None
        info['response_feature_cols'] = feature_cols
        if np.isfinite(fused_score) and (not np.isfinite(base_score) or fused_score >= base_score + float(getattr(cfg, 'fusion_min_improvement', -0.005))):
            val_df = val_df.copy()
            test_df = test_df.copy()
            val_df['response_pred_pre_fusion'] = val_df['response_pred']
            test_df['response_pred_pre_fusion'] = test_df['response_pred']
            _, Xv_all, _ = _prepare_fusion_matrix(val_clean, val_df, feature_cols)
            val_all_fused = model.predict(Xv_all)
            if bool(getattr(cfg, 'clip_auc_predictions', True)):
                val_all_fused = np.clip(val_all_fused, 0.0, 1.0)
            val_df['response_pred_fusion'] = val_all_fused
            test_df['response_pred_fusion'] = test_fused
            val_df['response_pred'] = val_df['response_pred_fusion']
            test_df['response_pred'] = test_df['response_pred_fusion']
            info['used_response_fusion'] = True
            info['response_medians'] = med
    if {'ln_ic50_true', 'ln_ic50_pred', 'ln_ic50_mask'}.issubset(val_df.columns):
        val_ic = val_df[(val_df['ln_ic50_mask'].astype(float) > 0) & val_df['ln_ic50_true'].notna() & val_df['ln_ic50_pred'].notna()].copy()
        if len(val_ic) >= min_rows:
            Xv, Xt, med = _prepare_fusion_matrix(val_ic, test_df, feature_cols)
            yv = val_ic['ln_ic50_true'].astype(float).values
            model = Ridge(alpha=float(getattr(cfg, 'fusion_alpha', 1.0)))
            model.fit(Xv, yv)
            val_fused = model.predict(Xv)
            test_fused = model.predict(Xt)
            base_metrics = ln_ic50_metrics_from_df(val_ic.assign(ln_ic50_pred=val_ic['ln_ic50_pred'].values))
            fused_metrics = ln_ic50_metrics_from_df(val_ic.assign(ln_ic50_pred=val_fused))
            base_score = base_metrics.get('ln_ic50_discovery_composite', -np.inf)
            fused_score = fused_metrics.get('ln_ic50_discovery_composite', -np.inf)
            info['ln_ic50_base_val_score'] = float(base_score) if np.isfinite(base_score) else None
            info['ln_ic50_fused_val_score'] = float(fused_score) if np.isfinite(fused_score) else None
            if np.isfinite(fused_score) and (not np.isfinite(base_score) or fused_score >= base_score + float(getattr(cfg, 'fusion_min_improvement', -0.005))):
                val_df = val_df.copy()
                test_df = test_df.copy()
                val_df['ln_ic50_pred_pre_fusion'] = val_df['ln_ic50_pred']
                test_df['ln_ic50_pred_pre_fusion'] = test_df['ln_ic50_pred']
                _, Xv_all, _ = _prepare_fusion_matrix(val_ic, val_df, feature_cols)
                val_df['ln_ic50_pred_fusion'] = model.predict(Xv_all)
                test_df['ln_ic50_pred_fusion'] = test_fused
                val_df['ln_ic50_pred'] = val_df['ln_ic50_pred_fusion']
                test_df['ln_ic50_pred'] = test_df['ln_ic50_pred_fusion']
                info['used_ln_ic50_fusion'] = True
                info['ln_ic50_medians'] = med
    return (val_df, test_df, info)

def run_druxr_v5(cfg: Optional[ConfigDruXRV5]=None, all_df: Optional[pd.DataFrame]=None, external_df: Optional[pd.DataFrame]=None, specialists: Tuple[str, ...]=('response_auc', 'response_ln_ic50', 'tic', 'biology', 'selectivity')):
    if cfg is None:
        cfg = CFG_DRUXR_V5
    _ensure_v5_config_defaults(cfg)
    root_output_dir = os.path.join(cfg.output_dir, cfg.run_name)
    ensure_dir(root_output_dir)
    print('\n' + '#' * 110)
    print('DruXR v5 CLEAN GRAPH-ONLY DUAL-RESPONSE MULTI-SPECIALIST SYSTEM')
    print('#' * 110)
    _print_df_block('SPECIALIST PLAN', specialist_design_table())
    specialist_results: Dict[str, Dict[str, Any]] = {}
    for specialist in specialists:
        if specialist not in SPECIALIST_SPECS:
            raise ValueError(f"Unknown specialist '{specialist}'. Available: {list(SPECIALIST_SPECS)}")
        print('\n' + '#' * 110)
        print(f'TRAINING SPECIALIST: {specialist}')
        print(f"Heads / outputs: {SPECIALIST_SPECS[specialist]['heads_outputs']}")
        print(f"Purpose: {SPECIALIST_SPECS[specialist]['purpose']}")
        print(f"Monitor: {SPECIALIST_SPECS[specialist]['monitor_metric']} ({SPECIALIST_SPECS[specialist]['monitor_mode']})")
        print('Weights:', SPECIALIST_SPECS[specialist]['weights'])
        print('#' * 110)
        sp_cfg = _make_specialist_cfg(cfg, specialist, root_output_dir)
        result = _run_single_specialist_base(cfg=sp_cfg, all_df=all_df)
        result['specialist'] = specialist
        result['specialist_spec'] = SPECIALIST_SPECS[specialist]
        specialist_results[specialist] = result
    overview = print_multispecialist_metric_report(specialist_results, root_output_dir)
    combined_predictions = build_multispecialist_combined_predictions(specialist_results)
    combined_val_predictions = getattr(build_multispecialist_combined_predictions, 'last_val_predictions', pd.DataFrame())
    fusion_info = getattr(build_multispecialist_combined_predictions, 'fusion_info', {})
    if not combined_predictions.empty:
        combined_predictions.to_csv(os.path.join(root_output_dir, 'multispecialist_combined_test_predictions.csv'), index=False)
    if isinstance(combined_val_predictions, pd.DataFrame) and (not combined_val_predictions.empty):
        combined_val_predictions.to_csv(os.path.join(root_output_dir, 'multispecialist_combined_val_predictions.csv'), index=False)
    with open(os.path.join(root_output_dir, 'multispecialist_fusion_info.json'), 'w') as f:
        json.dump(fusion_info, f, indent=2, default=str)
    combined_result_for_discovery = {'seed_results': {}, 'output_dir': root_output_dir, 'config': cfg}
    if not combined_predictions.empty and 'seed' in combined_predictions.columns:
        group_cols = ['split_seed', 'seed'] if 'split_seed' in combined_predictions.columns else ['seed']
        for group_key, sub in combined_predictions.groupby(group_cols):
            if isinstance(group_key, tuple):
                split_seed, model_seed = (int(group_key[0]), int(group_key[1]))
            else:
                split_seed, model_seed = (int(sub['split_seed'].iloc[0]) if 'split_seed' in sub.columns else 1, int(group_key))
            run_key = f'split{split_seed}_seed{model_seed}'
            combined_result_for_discovery['seed_results'][run_key] = {'test_predictions': sub.copy(), 'split_seed': split_seed, 'model_seed': model_seed}
    discovery_reports = {}
    if combined_result_for_discovery['seed_results']:
        try:
            discovery_reports = print_discovery_top10_report(result=combined_result_for_discovery, all_df=all_df, external_df=external_df, output_dir=root_output_dir, top_n=int(getattr(cfg, 'discovery_top_n', 10)))
        except Exception as e:
            print('\n[WARNING] Discovery report failed after training.')
            print('Training outputs and combined predictions were already saved.')
            print('Error:', repr(e))
            discovery_reports = {'error': repr(e)}
    print('\n' + '#' * 110)
    print('MULTI-SPECIALIST RUN COMPLETE')
    print('Artifacts saved under:', root_output_dir)
    print('Key files:')
    print(' - multispecialist_overview_metrics.csv')
    print(' - multispecialist_combined_test_predictions.csv')
    print(' - multispecialist_combined_val_predictions.csv')
    print(' - multispecialist_fusion_info.json')
    print(' - discovery_report/discovery_top10_all_categories.csv')
    print('#' * 110)
    return {'specialist_results': specialist_results, 'summary': overview, 'combined_predictions': combined_predictions, 'combined_val_predictions': combined_val_predictions, 'fusion_info': fusion_info, 'discovery_reports': discovery_reports, 'output_dir': root_output_dir, 'config': cfg, 'specialist_design': specialist_design_table()}

def rerun_discovery_from_saved_predictions_v5_r2(all_df: pd.DataFrame, cfg: ConfigDruXRV5=CFG_DRUXR_V5, external_df: Optional[pd.DataFrame]=None, root_output_dir: Optional[str]=None, top_n: Optional[int]=None) -> Dict[str, pd.DataFrame]:
    if root_output_dir is None:
        root_output_dir = os.path.join(cfg.output_dir, cfg.run_name)
    if top_n is None:
        top_n = int(getattr(cfg, 'discovery_top_n', 10))
    pred_path = os.path.join(root_output_dir, 'multispecialist_combined_test_predictions.csv')
    if not os.path.exists(pred_path):
        raise FileNotFoundError(f'Saved prediction file not found: {pred_path}')
    combined_predictions = pd.read_csv(pred_path)
    combined_result_for_discovery = {'seed_results': {}, 'output_dir': root_output_dir, 'config': cfg}
    if 'seed' not in combined_predictions.columns:
        combined_predictions['seed'] = int(getattr(cfg, 'seed', 1))
    group_cols = ['split_seed', 'seed'] if 'split_seed' in combined_predictions.columns else ['seed']
    for group_key, sub in combined_predictions.groupby(group_cols, dropna=False):
        if isinstance(group_key, tuple):
            split_seed = int(group_key[0])
            model_seed = int(group_key[1])
        else:
            split_seed = int(sub['split_seed'].iloc[0]) if 'split_seed' in sub.columns else 1
            model_seed = int(group_key)
        run_key = f'split{split_seed}_seed{model_seed}'
        combined_result_for_discovery['seed_results'][run_key] = {'test_predictions': sub.copy(), 'split_seed': split_seed, 'model_seed': model_seed}
    return print_discovery_top10_report(result=combined_result_for_discovery, all_df=all_df, external_df=external_df, output_dir=root_output_dir, top_n=top_n)
print('Loaded DruXR v5 clean graph-only dual-response multi-specialist system.')
print("Then inspect: result_v5['summary'], result_v5['combined_predictions'], result_v5['fusion_info'], result_v5['discovery_reports']")
print('Saved-output folder:', os.path.join(CFG_DRUXR_V5.output_dir, CFG_DRUXR_V5.run_name))
result_v5 = run_druxr_v5(CFG_DRUXR_V5, all_df=all_merged_full)
result_v5['summary']
